# TradeSense · Component 03 · Layer 2 → `processed_layer2_model_ready_v1`

**Scope: Layer 2 only** (Sapienza pump-and-dump, Mt.Gox wash-trading). No Layer 1/3, no Autoencoder/Transformer/TGNN, **no model training**.
**Frozen inputs (never modified):** `processed_layer2_final_v3/` and the two raw Layer 2 sources. **Output:** `processed_layer2_model_ready_v1/` (new, versioned; the notebook refuses to run if it already exists).

## What this notebook actually does
| Task | Executable action | Decision rule (fixed in advance, no test labels used) |
|---|---|---|
| **A** Sapienza sidecars | Replays the verified split on the raw CSV (carrying an explicit `source_row_id`), **asserts bit-for-bit equality** with the frozen parquets, writes row-aligned train/val/test sidecars (timestamp, coin, `pump_index`, event group, label, source row id, missingness, duplicate/coin flags). | Hard assertions; any mismatch aborts. |
| **B** Sapienza duplicates | Classifies every repeated-vector row (legitimate inactive background / imputation-driven / potentially harmful / cross-split positive). | A training-only filter is created **only** for rows that are *identical source records* (all raw columns incl. timestamp and coin). Nothing else is removed. The primary benchmark stays unfiltered. |
| **C** Coin identity | Coin overlap per split; coin-unseen evaluation subsets; coin-grouped CV folds **inside train**; identity probe (η² and a nearest-centroid probe, **train-fitted only**); feature configs **A** (original) and **B** (price-identity-reduced). | B drops a feature only if it is an absolute price-level feature **and** train η²(coin) ≥ 0.25. Final A-vs-B choice is left to *validation* in the XGBoost notebook. |
| **D** Imbalance | Class weights from TRAIN labels only (full, sqrt, none), per-fold weights, rare-event metric specification + a tested metrics module (`layer2_eval_utils.py`). | No resampling, no SMOTE. |
| **E/F** Mt.Gox | Reads **all** source columns, documentation, time/entity/label-derivation evidence, within-conflict analysis, reproduces the frozen split, writes row flags, classifies the benchmark **A / B / C from evidence**. | Explicit, printed rule set; never forced to PASS. |
| **G** Artifacts | Staged write → manifests (loadable without guessing) → README. | Frozen parquets are *referenced* (path + SHA-256), not copied. |
| **H** Validation | Reads the staged artifacts back and asserts alignment, consistency, no refit, class weights, fold independence, frozen-file hashes, readability, and unit-tests the metrics module. | **Publish (atomic rename) only if every assertion passes.** |

**Runtime:** ≈ 5–15 min on a Colab CPU runtime; peak RAM ≈ 3–5 GB (Mt.Gox source). **Never overwrites:** output is staged in a hidden temp directory inside the dataset root and renamed only at the very end.

## 0 · Setup, configuration and frozen-input pinning

In [ ]:
import os, sys, json, time, math, re, hashlib, shutil, uuid, warnings
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy import stats
import sklearn, joblib
from IPython.display import display

warnings.filterwarnings('ignore', category=FutureWarning); warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 60); pd.set_option('display.max_rows', 120)
pd.set_option('display.float_format', lambda v: f'{v:,.6g}')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> set TRADESENSE_BASE to your local dataset root.')

BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
assert BASE.is_dir(), f'Dataset root not found: {BASE}'
L2_ROOT   = BASE / 'processed_layer2_final_v3'                                          # FROZEN
OUT_FINAL = BASE / 'processed_layer2_model_ready_v1'                                    # NEW
STAGE     = BASE / ('.layer2_model_ready_stage_' + uuid.uuid4().hex[:8])               # hidden staging dir (same filesystem => atomic rename)
SAP_DIR   = BASE / 'pump-and-dump-dataset-master'
SAP_FILE  = SAP_DIR / 'labeled_features/features_5S.csv/features_5S.csv'
GOX_FILE  = BASE / 'gox_ml_samples.csv'
assert L2_ROOT.is_dir(), f'Frozen Layer 2 directory missing: {L2_ROOT}'
assert not OUT_FINAL.exists(), f'REFUSING to overwrite existing output: {OUT_FINAL}'
for p in (SAP_FILE, GOX_FILE): assert p.is_file(), f'Source missing: {p}'
SEED = 42; RUN_STARTED = datetime.now(timezone.utc).isoformat()
print('base:', BASE, '\nfrozen:', L2_ROOT, '\noutput (new):', OUT_FINAL, '\nstage:', STAGE.name)
print('sklearn', sklearn.__version__, '| pandas', pd.__version__, '| numpy', np.__version__, '| seed', SEED)

# ---------------- feature definitions copied from 02_layer2_final_pipeline_and_audit.ipynb ----------------
SAP_BASE = ['std_rush_order', 'avg_rush_order', 'std_trades', 'std_volume', 'avg_volume', 'std_price', 'avg_price', 'avg_price_max', 'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos']
GOX_BASE = ['bitcoins', 'price_deviation', 'buyer_24h_trade_count', 'seller_24h_trade_count']
EMBARGO  = pd.Timedelta(hours=1)
SAP_ACTIVITY = ['std_rush_order', 'avg_rush_order', 'std_trades', 'std_volume', 'avg_volume', 'std_price']

# ---------------- pre-registered decision constants ----------------
CFG = {'eta2_drop': 0.25,            # config B: drop absolute price-level features whose train eta^2(coin) >= this
       'coin_folds': 5, 'min_coin_rows_probe': 30, 'probe_eval_rows': 20_000,
       'sample_auc_rows': 400_000, 'auc_deterministic_dev': 0.49, 'auc_explains_conflict_dev': 0.20,
       'mtgox_blocks': 10, 'mono_fraction': 0.999, 'time_min_unique': 1000,
       'gox_conflict_floor_unsuitable': 0.30, 'gox_repeat_share_unsuitable': 0.50, 'gox_conflict_floor_qualified': 0.10,
       'min_pos_groups': 5}

Mounted at /content/drive
base: /content/drive/MyDrive/research datasets 
frozen: /content/drive/MyDrive/research datasets/processed_layer2_final_v3 
output (new): /content/drive/MyDrive/research datasets/processed_layer2_model_ready_v1 
stage: .layer2_model_ready_stage_1531c7a1
sklearn 1.6.1 | pandas 2.2.3 | numpy 2.1.3 | seed 42


In [ ]:
def sha256_file(path, chunk=8 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()
def hash_rows(df, cols): return pd.util.hash_pandas_object(df[cols], index=False).to_numpy()
def jsonable(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (pd.Timestamp, datetime)): return o.isoformat()
    if isinstance(o, Path): return str(o)
    return str(o)
def dump_json(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True); path.write_text(json.dumps(obj, indent=2, default=jsonable), encoding='utf-8')

def fit_params_numpy(x_df):
    """numpy re-implementation of SimpleImputer(median)+RobustScaler fitting statistics (nothing fitted/saved)."""
    X = x_df.to_numpy(float); med = np.nanmedian(X, axis=0); med = np.where(np.isnan(med), 0.0, med)
    Xi = np.where(np.isnan(X), med, X); center = np.median(Xi, axis=0); q = np.percentile(Xi, [25, 75], axis=0); scale = q[1] - q[0]
    return med, center, np.where(scale < 10 * np.finfo(float).eps, 1.0, scale)
def max_rel_diff(saved, cand):
    return float(max(np.max(np.abs(np.asarray(a, float) - b) / np.maximum(np.abs(np.asarray(a, float)), 1e-12)) for a, b in zip(saved, cand)))

def scan_docs(root, max_depth=3, max_files=6, max_bytes=400_000):
    found = []
    if not root.is_dir(): return found
    d0 = len(root.parts)
    for dp, dn, fn in os.walk(root):
        if len(Path(dp).parts) - d0 >= max_depth: dn[:] = []
        for f in sorted(fn):
            if re.search(r'readme|licen[cs]e|citation|\.md$|\.rst$|\.txt$', f, re.I):
                p = Path(dp) / f
                try:
                    if p.stat().st_size <= max_bytes: found.append(p)
                except OSError: pass
        if len(found) >= max_files: break
    return found[:max_files]

# ---------------- pin the FROZEN inputs (size, mtime, SHA-256) ----------------
t0 = time.time()
FROZEN_FILES = sorted(p for p in L2_ROOT.iterdir() if p.is_file())
FROZEN_PIN = {p.name: {'size': p.stat().st_size, 'mtime_ns': p.stat().st_mtime_ns, 'sha256': sha256_file(p)} for p in FROZEN_FILES}
SOURCE_PIN = {p.name: {'path': str(p), 'size': p.stat().st_size, 'mtime_ns': p.stat().st_mtime_ns, 'sha256': sha256_file(p)} for p in (SAP_FILE, GOX_FILE)}
FROZEN_LISTING = sorted(p.name for p in L2_ROOT.iterdir())
print(f'Pinned {len(FROZEN_PIN)} frozen v3 files + 2 raw sources in {time.time()-t0:.0f}s')
display(pd.DataFrame(FROZEN_PIN).T[['size', 'sha256']].assign(sha256=lambda d: d.sha256.str[:16] + '…'))

Pinned 8 frozen v3 files + 2 raw sources in 23s


,size,sha256
mtgox_metadata.joblib,1897,109f10b9121a9d64…
mtgox_test.parquet,14033431,227f9df040c72838…
mtgox_train.parquet,62728544,5637904dd437c476…
mtgox_val.parquet,13836494,5289c23eb7ac13a4…
sapienza_metadata.joblib,2368,83a8b98fabbc4a13…
sapienza_test.parquet,723461,773e5fcfcb1f47fb…
sapienza_train.parquet,4736194,a79c5e7c569790ca…
sapienza_val.parquet,932166,bea3b17c72298d20…


## A · Sapienza: replay → alignment proof → row-aligned sidecars
The replay below is the published `sapienza_split` logic copied verbatim, with one addition: an explicit `source_row_id` (original CSV row position) is carried through the stable sort so every retained row keeps its provenance. The result must equal the frozen parquets **bit-for-bit** or the notebook aborts.

In [ ]:
SAP_M = joblib.load(L2_ROOT / 'sapienza_metadata.joblib'); SAP_FEATS = list(SAP_M['features']); SAP_T = 'gt'
SAP_PQ = {sp: pd.read_parquet(L2_ROOT / f'sapienza_{sp}.parquet') for sp in ('train', 'val', 'test')}
SPL = ('train', 'val', 'test')
assert list(SAP_PQ['train'].columns) == SAP_FEATS + [SAP_T], 'frozen parquet column order != metadata feature order + target'
print('frozen rows:', {k: len(v) for k, v in SAP_PQ.items()}, '| features:', len(SAP_FEATS))

src = pd.read_csv(SAP_FILE)                                          # ALL columns (needed for exact source-record duplicate detection)
SRC_COLS = list(src.columns)
src['source_row_id'] = np.arange(len(src), dtype=np.int64)
print(f'source rows {len(src):,}; columns ({len(SRC_COLS)}):', SRC_COLS)

def sap_features(df):
    x = df[SAP_BASE].apply(pd.to_numeric, errors='coerce').copy()
    x['rush_volume_interaction'] = x['avg_rush_order'] * x['avg_volume']
    x['log_avg_volume'] = np.log1p(x['avg_volume'].clip(lower=0)); x['log_std_price'] = np.log1p(x['std_price'].clip(lower=0))
    return x.replace([np.inf, -np.inf], np.nan)

def replay_sapienza_split(df):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'], utc=True, errors='coerce'); df['pump_index'] = pd.to_numeric(df['pump_index'], errors='coerce')
    if df['date'].isna().any() or not df['gt'].isin([0, 1]).all(): raise ValueError('Sapienza invalid date or target')
    if (df.loc[df['gt'] == 1, 'pump_index'].isna() | df.loc[df['gt'] == 1, 'pump_index'].lt(0)).any(): raise ValueError('positives without nonnegative pump_index')
    df = df.sort_values('date', kind='mergesort').reset_index(drop=True)
    event = df['pump_index'].ge(0)
    groups = df.loc[event].groupby('pump_index')['date'].agg(start='min', end='max', n='size').sort_values(['start', 'end'])
    if len(groups) < 20: raise ValueError('Too few event groups')
    cut1 = groups['start'].iloc[int(len(groups) * .70)]; cut2 = groups['start'].iloc[int(len(groups) * .85)]
    lo1, hi1, lo2, hi2 = cut1 - EMBARGO, cut1 + EMBARGO, cut2 - EMBARGO, cut2 + EMBARGO
    tr = set(groups.index[groups['end'] < lo1]); va = set(groups.index[(groups['start'] >= hi1) & (groups['end'] < lo2)]); te = set(groups.index[groups['start'] >= hi2])
    bg = ~event
    parts = {'train': df.loc[(event & df.pump_index.isin(tr)) | (bg & df.date.lt(lo1))].copy(),
             'val':   df.loc[(event & df.pump_index.isin(va)) | (bg & df.date.ge(hi1) & df.date.lt(lo2))].copy(),
             'test':  df.loc[(event & df.pump_index.isin(te)) | (bg & df.date.ge(hi2))].copy()}
    return parts, groups, {'cut1': cut1, 'cut2': cut2, 'n_source_rows': len(df)}

SAP_PARTS, SAP_GROUPS, SAP_INFO = replay_sapienza_split(src)
SAP_IMP, SAP_SC = SAP_M['imputer'], SAP_M['scaler']
ALIGN, SAP_X_RAW = {}, {}
for sp in SPL:
    part = SAP_PARTS[sp].reset_index(drop=True); SAP_PARTS[sp] = part
    SAP_X_RAW[sp] = sap_features(part)
    arr = SAP_SC.transform(SAP_IMP.transform(SAP_X_RAW[sp])).astype(np.float32); saved = SAP_PQ[sp][SAP_FEATS].to_numpy(np.float32)
    ALIGN[sp] = {'rows_replayed': len(part), 'rows_saved': len(saved), 'features_bit_identical': bool(len(arr) == len(saved) and np.array_equal(arr, saved)),
                 'labels_identical': bool(np.array_equal(part[SAP_T].to_numpy().astype(np.int8), SAP_PQ[sp][SAP_T].to_numpy().astype(np.int8))),
                 'dates_non_decreasing': bool(part['date'].is_monotonic_increasing)}
display(pd.DataFrame(ALIGN).T)
for sp, a in ALIGN.items():
    assert a['rows_replayed'] == a['rows_saved'] and a['features_bit_identical'] and a['labels_identical'] and a['dates_non_decreasing'], f'ALIGNMENT FAILED for {sp}: {a}'
ids = {sp: set(SAP_PARTS[sp].loc[SAP_PARTS[sp].pump_index >= 0, 'pump_index']) for sp in SPL}
assert not (ids['train'] & ids['val'] or ids['train'] & ids['test'] or ids['val'] & ids['test']), 'pump_index groups shared across splits'
assert SAP_PARTS['train'].date.max() < SAP_PARTS['val'].date.min() <= SAP_PARTS['val'].date.max() < SAP_PARTS['test'].date.min(), 'splits are not chronologically separated'
gap = {'train->val': str(SAP_PARTS['val'].date.min() - SAP_PARTS['train'].date.max()), 'val->test': str(SAP_PARTS['test'].date.min() - SAP_PARTS['val'].date.max())}
assert SAP_PARTS['val'].date.min() - SAP_PARTS['train'].date.max() >= EMBARGO and SAP_PARTS['test'].date.min() - SAP_PARTS['val'].date.max() >= EMBARGO
# train-only fitting evidence (independent numpy recomputation from the replayed raw train partition)
p_train = max_rel_diff((SAP_IMP.statistics_, SAP_SC.center_, SAP_SC.scale_), fit_params_numpy(SAP_X_RAW['train']))
p_all = max_rel_diff((SAP_IMP.statistics_, SAP_SC.center_, SAP_SC.scale_), fit_params_numpy(pd.concat(SAP_X_RAW.values())))
assert p_train < 1e-6, f'saved imputer/scaler != train-only recomputation ({p_train:.2e})'
SAP_FACTS = {'alignment': ALIGN, 'split_gaps': gap, 'param_rel_diff_train_only': p_train, 'param_rel_diff_all_partitions': p_all,
             'positive_rows': {sp: int(SAP_PQ[sp][SAP_T].sum()) for sp in SPL},
             'event_groups': {sp: len(ids[sp]) for sp in SPL},
             'positive_groups': {sp: int(SAP_PARTS[sp].loc[SAP_PARTS[sp]['gt'] == 1, 'pump_index'].nunique()) for sp in SPL},
             'source_rows': SAP_INFO['n_source_rows'], 'rows_retained': sum(len(p) for p in SAP_PARTS.values())}
print('ALIGNMENT PROVEN (bit-for-bit) | train-only fit proven (rel diff %.1e vs all-partition %.1e)' % (p_train, p_all)); print(json.dumps(SAP_FACTS, indent=1, default=jsonable)[:1800])

frozen rows: {'train': 600827, 'val': 121255, 'test': 89334} | features: 15
source rows 821,307; columns (16): ['date', 'pump_index', 'std_rush_order', 'avg_rush_order', 'std_trades', 'std_volume', 'avg_volume', 'std_price', 'avg_price', 'avg_price_max', 'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos', 'symbol', 'gt']


,rows_replayed,rows_saved,features_bit_identical,labels_identical,dates_non_decreasing
train,600827,600827,True,True,True
val,121255,121255,True,True,True
test,89334,89334,True,True,True


ALIGNMENT PROVEN (bit-for-bit) | train-only fit proven (rel diff 0.0e+00 vs all-partition 1.0e+03)
{
 "alignment": {
  "train": {
   "rows_replayed": 600827,
   "rows_saved": 600827,
   "features_bit_identical": true,
   "labels_identical": true,
   "dates_non_decreasing": true
  },
  "val": {
   "rows_replayed": 121255,
   "rows_saved": 121255,
   "features_bit_identical": true,
   "labels_identical": true,
   "dates_non_decreasing": true
  },
  "test": {
   "rows_replayed": 89334,
   "rows_saved": 89334,
   "features_bit_identical": true,
   "labels_identical": true,
   "dates_non_decreasing": true
  }
 },
 "split_gaps": {
  "train->val": "2 days 23:24:15",
  "val->test": "6 days 00:23:55"
 },
 "param_rel_diff_train_only": 0.0,
 "param_rel_diff_all_partitions": 999.4999167083071,
 "positive_rows": {
  "train": 220,
  "val": 44,
  "test": 48
 },
 "event_groups": {
  "train": 225,
  "val": 48,
  "test": 49
 },
 "positive_groups": {
  "train": 220,
  "val": 44,
  "test": 48
 },
 "source

In [ ]:
# ---------------- coin / identifier columns: by NAME only; nothing is assumed ----------------
USED = set(['date', 'pump_index', 'gt', 'source_row_id'] + SAP_BASE)
ID_COLS = [c for c in SRC_COLS if c not in USED and re.search(r'coin|symbol|pair|currency|ticker|asset|market|exchange|channel', str(c), re.I)]
EXTRA_COLS = [c for c in SRC_COLS if c not in USED]
print('columns not used as features:', EXTRA_COLS); print('identifier-like columns (name match):', ID_COLS)
for c in ID_COLS: print(f'  {c}: dtype={src[c].dtype}, n_unique={src[c].nunique()}, top={src[c].value_counts().head(3).to_dict()}')
COIN_PRIORITY = ['symbol', 'coin', 'ticker', 'currency', 'pair', 'asset', 'market']
PRIMARY_COIN = next((c for tok in COIN_PRIORITY for c in ID_COLS if re.search(tok, str(c), re.I)), None)
print('PRIMARY coin identifier column:', PRIMARY_COIN or 'NONE FOUND -> coin analysis will be documented as unavailable')

src_nid = src.drop(columns=['source_row_id'])
SREC_HASH = pd.util.hash_pandas_object(src_nid, index=False).to_numpy(); del src_nid           # identical-SOURCE-RECORD hash (all raw columns incl. timestamp and coin)
SRC_DATE_CHECK = src[['date', 'pump_index', 'gt']].copy()                                          # kept for the sidecar read-back validation
SRC_N = len(src); del src

columns not used as features: ['symbol']
identifier-like columns (name match): ['symbol']
  symbol: dtype=object, n_unique=85, top={'DLT': 47544, 'EVX': 32664, 'NAV': 31029}
PRIMARY coin identifier column: symbol


## B · Sapienza duplicates: classify, decide, document (no deletion of legitimate observations)
Every row whose scaled feature vector occurs ≥ 2 times (across all three splits) is assigned **one** category, in this priority order:

| Code | Meaning | Treatment |
|---|---|---|
| **D** | positive row whose vector also occurs in another split | none expected; reported (would be the clearest leakage) |
| **B** | event/positive row, or a cluster mixing labels, or an **identical source record** (all raw columns, incl. timestamp & coin) | potentially harmful; only *identical source records* are candidates for the training-only filter |
| **C** | repeated vector in which the row has ≥ 1 imputed (originally missing) feature | imputation-driven; **retained** |
| **A** | repeated background vector with all activity features 0/missing | inactive window; legitimate; **retained** |
| **A2** | other repeated background vector | retained, reported |

In [ ]:
frames = []
for i, sp in enumerate(SPL):
    part, x = SAP_PARTS[sp], SAP_X_RAW[sp]
    frames.append(pd.DataFrame({'split': sp, 'split_code': i, 'row_id': np.arange(len(part), dtype=np.int32), 'source_row_id': part['source_row_id'].to_numpy(np.int64),
                                'h': hash_rows(SAP_PQ[sp], SAP_FEATS), 'gt': SAP_PQ[sp][SAP_T].to_numpy().astype(np.int8), 'pump_index': part['pump_index'].to_numpy(float),
                                'n_missing': x.isna().sum(axis=1).to_numpy(np.int8), 'inactive': (x[SAP_ACTIVITY].fillna(0) == 0).all(axis=1).to_numpy(),
                                'coin': part[PRIMARY_COIN].astype('string').to_numpy() if PRIMARY_COIN else None,
                                'srec_h': SREC_HASH[part['source_row_id'].to_numpy(np.int64)]}))
G = pd.concat(frames, ignore_index=True); del frames
G['is_event'] = G.pump_index >= 0
g = G.groupby('h')
G['cluster_n'] = g['h'].transform('size'); G['cluster_pos'] = g['gt'].transform('sum'); G['cluster_nsplits'] = g['split_code'].transform('nunique'); G['cluster_event'] = g['is_event'].transform('max')
G['cluster_mixed'] = (G.cluster_pos > 0) & (G.cluster_pos < G.cluster_n)
gs = G.groupby('srec_h')
G['srec_n'] = gs['srec_h'].transform('size'); G['srec_rank'] = gs.cumcount(); G['srec_nsplits'] = gs['split_code'].transform('nunique')   # G order = train, val, test (chronological within split)
G['srec_in_eval'] = gs['split_code'].transform('max') > 0
rep = G.cluster_n >= 2
G['dup_category'] = np.select([~rep,
                               rep & (G['gt'] == 1) & (G.cluster_nsplits > 1),
                               rep & ((G['gt'] == 1) | G.is_event | G.cluster_mixed | G.cluster_event.astype(bool) | (G.srec_n >= 2)),
                               rep & (G.n_missing >= 1),
                               rep & G.inactive,
                               rep], ['none', 'D_cross_split_positive', 'B_potentially_harmful', 'C_imputation_driven', 'A_inactive_background', 'A2_other_background_repeat'], default='none')
tab = G.groupby(['dup_category', 'split']).size().unstack(fill_value=0).reindex(columns=list(SPL), fill_value=0); tab['total'] = tab.sum(axis=1)
print('Repeated-vector rows by category and split:'); display(tab)
pos_tab = G[G['gt'] == 1].groupby(['dup_category', 'split']).size().unstack(fill_value=0); print('POSITIVE rows by category:'); display(pos_tab)

# ---- cross-split leakage facts ----
trh = set(G.loc[G.split == 'train', 'h'])
LEAK = {}
for sp in ('val', 'test'):
    e = G[G.split == sp]; m = e.h.isin(trh)
    LEAK[sp] = {'rows': len(e), 'rows_vector_in_train': int(m.sum()), 'share_vector_in_train': float(m.mean()), 'positives': int(e['gt'].sum()), 'positives_vector_in_train': int((m & (e['gt'] == 1)).sum()),
                'rows_identical_source_record_in_train': int(e.srec_h.isin(set(G.loc[G.split == 'train', 'srec_h'])).sum())}
trl = G[G.split == 'train'].groupby('h')['gt'].agg(['min', 'max'])
for sp in ('val', 'test'):
    e = G[G.split == sp].join(trl, on='h', how='inner'); LEAK[sp]['dup_rows_label_conflict_with_train'] = int(((e['min'] != e['gt']) | (e['max'] != e['gt'])).sum())
print(json.dumps(LEAK, indent=1))

Repeated-vector rows by category and split:


split,train,val,test,total
dup_category,,,,
B_potentially_harmful,189224,39536,24647,253407
none,411603,81719,64687,558009


POSITIVE rows by category:


split,test,train,val
dup_category,,,
none,48,220,44


{
 "val": {
  "rows": 121255,
  "rows_vector_in_train": 30996,
  "share_vector_in_train": 0.2556265721001196,
  "positives": 44,
  "positives_vector_in_train": 0,
  "rows_identical_source_record_in_train": 0,
  "dup_rows_label_conflict_with_train": 0
 },
 "test": {
  "rows": 89334,
  "rows_vector_in_train": 18448,
  "share_vector_in_train": 0.2065059215976,
  "positives": 48,
  "positives_vector_in_train": 0,
  "rows_identical_source_record_in_train": 0,
  "dup_rows_label_conflict_with_train": 0
 }
}


In [ ]:
# ---------------- evidence-based exclusion decision (identical SOURCE RECORDS only; label-free; train rows only) ----------------
tr_mask = (G.split == 'train')
R1 = tr_mask & (G.srec_n >= 2) & (G.srec_rank > 0)                              # later copy of an identical source record inside train
R2 = tr_mask & (G.srec_n >= 2) & (G.srec_rank == 0) & G.srec_in_eval            # train copy of a record that also occurs in val/test (decontamination)
EXCL = G[R1 | R2][['row_id', 'source_row_id', 'srec_h']].copy(); EXCL['reason'] = np.where(R1[R1 | R2], 'R1_train_internal_identical_source_record', 'R2_train_copy_of_record_present_in_eval')
EXCL['pump_index'] = G.loc[EXCL.index, 'pump_index']; EXCL['gt'] = G.loc[EXCL.index, 'gt']
n_tr = int(tr_mask.sum()); y_tr = G.loc[tr_mask, 'gt']
before = {'rows': n_tr, 'positives': int(y_tr.sum()), 'negatives': int((1 - y_tr).sum())}
keep_mask = ~(R1 | R2)[tr_mask].to_numpy()
y_after = y_tr.to_numpy()[keep_mask]
after = {'rows': int(keep_mask.sum()), 'positives': int(y_after.sum()), 'negatives': int((1 - y_after).sum())}
print('identical-source-record duplicates in the retained data:', int((G.srec_n >= 2).sum()), 'rows | candidates for training-only exclusion:', len(EXCL))
if len(EXCL): display(EXCL.groupby('reason').agg(rows=('row_id', 'size'), positives=('gt', 'sum')))
print('TRAIN before:', before, '| after filter option:', after)
FILTER_JUSTIFIED = len(EXCL) > 0
DUP_DECISION = {'primary_benchmark': 'UNFILTERED (frozen v3 train/val/test unchanged)', 'filter_option_created': FILTER_JUSTIFIED,
                'filter_rule': 'exclude TRAIN rows that are identical source records (all raw CSV columns incl. timestamp and coin): later copies inside train (R1) and train copies of records that also occur in val/test (R2). Uses no labels.',
                'train_before': before, 'train_after_filter': after, 'rows_excluded': int(len(EXCL)), 'excluded_positives': int(EXCL['gt'].sum()) if len(EXCL) else 0,
                'not_removed': 'repeated vectors that are inactive/imputed background (categories A, A2, C) - legitimate repeated observations; validation/test rows - never filtered',
                'remaining_cross_split_leakage': LEAK,
                'category_counts': tab.reset_index().to_dict(orient='records'), 'positive_category_counts': pos_tab.reset_index().to_dict(orient='records')}
if not FILTER_JUSTIFIED: DUP_DECISION['why_no_exclusion'] = 'no identical source record occurs more than once among retained rows; repeated vectors are explained by inactivity/imputation or label-consistent background, so none is removed.'
print(json.dumps({k: v for k, v in DUP_DECISION.items() if k not in ('category_counts', 'positive_category_counts', 'remaining_cross_split_leakage')}, indent=1, default=jsonable))

identical-source-record duplicates in the retained data: 0 rows | candidates for training-only exclusion: 0
TRAIN before: {'rows': 600827, 'positives': 220, 'negatives': 600607} | after filter option: {'rows': 600827, 'positives': 220, 'negatives': 600607}
{
 "primary_benchmark": "UNFILTERED (frozen v3 train/val/test unchanged)",
 "filter_option_created": false,
 "filter_rule": "exclude TRAIN rows that are identical source records (all raw CSV columns incl. timestamp and coin): later copies inside train (R1) and train copies of records that also occur in val/test (R2). Uses no labels.",
 "train_before": {
  "rows": 600827,
  "positives": 220,
  "negatives": 600607
 },
 "train_after_filter": {
  "rows": 600827,
  "positives": 220,
  "negatives": 600607
 },
 "rows_excluded": 0,
 "excluded_positives": 0,
 "not_removed": "repeated vectors that are inactive/imputed background (categories A, A2, C) - legitimate repeated observations; validation/test rows - never filtered",
 "why_no_exclusion

## C · Sapienza coin identity: overlap, coin-unseen subsets, coin-grouped folds, identity probe, feature configs A/B
* **Official benchmark unchanged** (chronological, event-disjoint).
* **Coin-unseen subsets** (val/test rows whose coin never occurs in train) = the closest defensible *coin-generalisation test under the chronological protocol*.
* **Coin-grouped folds inside TRAIN** (all rows of a coin, and all events linked to it, in one fold) for coin-held-out model selection. A strict coin-disjoint re-split of the *whole* dataset would sacrifice the verified chronology/event-disjoint design, so it is **not** constructed.
* **Identity probe** uses **train rows only** (η² between coin and each scaled feature; a nearest-centroid coin classifier fitted on train and evaluated on *validation* rows — not XGBoost, not test).

In [ ]:
COIN = {'available': PRIMARY_COIN is not None, 'column': PRIMARY_COIN}
SAP_SIDECAR = {}
trc = set(G.loc[(G.split == 'train') & G.coin.notna(), 'coin']) if PRIMARY_COIN else set()
G['coin_seen_in_train'] = G.coin.isin(trc) if PRIMARY_COIN else pd.Series(False, index=G.index)
G['vec_seen_in_train'] = G.h.isin(trh) & (G.split != 'train')
if PRIMARY_COIN:
    rows = []
    for sp in SPL:
        e = G[G.split == sp]
        rows.append({'split': sp, 'rows': len(e), 'distinct_coins': int(e.coin.nunique()), 'rows_coin_seen_in_train': float(e.coin_seen_in_train.mean()),
                     'positives': int(e['gt'].sum()), 'positives_coin_seen': float(e.loc[e['gt'] == 1, 'coin_seen_in_train'].mean()) if e['gt'].sum() else np.nan,
                     'event_groups': int(e.loc[e.is_event, 'pump_index'].nunique()), 'event_groups_coin_seen': float(e[e.is_event].groupby('pump_index')['coin_seen_in_train'].first().mean()) if e.is_event.any() else np.nan,
                     'UNSEEN_rows': int((~e.coin_seen_in_train).sum()) if sp != 'train' else 0, 'UNSEEN_positives': int(((~e.coin_seen_in_train) & (e['gt'] == 1)).sum()) if sp != 'train' else 0,
                     'UNSEEN_positive_groups': int(e.loc[(~e.coin_seen_in_train) & (e['gt'] == 1), 'pump_index'].nunique()) if sp != 'train' else 0})
    COIN_TAB = pd.DataFrame(rows).set_index('split'); display(COIN_TAB); COIN['overlap'] = COIN_TAB.reset_index().to_dict(orient='records')
    # pump_index must map to exactly one coin for coin-grouped folds to also be event-disjoint
    multi = G[G.is_event & G.coin.notna()].groupby('pump_index')['coin'].nunique(); COIN['pump_indices_spanning_multiple_coins'] = int((multi > 1).sum())
    print('pump_index groups spanning >1 coin:', COIN['pump_indices_spanning_multiple_coins'])
else:
    print('No coin identifier column in the raw source -> coin-identity leakage cannot be tested; documented as a limitation.')

,rows,distinct_coins,rows_coin_seen_in_train,positives,positives_coin_seen,event_groups,event_groups_coin_seen,UNSEEN_rows,UNSEEN_positives,UNSEEN_positive_groups
split,,,,,,,,,,
train,600827,71,1,220,1,225,1,0,0,0
val,121255,36,0.770335,44,0.772727,48,0.791667,27848,10,10
test,89334,29,0.751953,48,0.729167,49,0.714286,22159,13,13


pump_index groups spanning >1 coin: 0


In [ ]:
# ---------------- identity probe on TRAIN only ----------------
SAP_X = {sp: SAP_PQ[sp][SAP_FEATS].to_numpy(np.float32) for sp in SPL}
ETA = None; PROBE = {}
PRICE_LEVEL = [f for f in SAP_FEATS if 'price' in f.lower()]
if PRIMARY_COIN:
    ctr = G.loc[G.split == 'train', 'coin'].to_numpy(); Xtr = SAP_X['train']
    d = pd.DataFrame(Xtr, columns=SAP_FEATS); d['coin'] = ctr; d = d[d.coin.notna()]
    cnt = d.groupby('coin').size(); mean_c = d.groupby('coin')[SAP_FEATS].mean(); grand = d[SAP_FEATS].mean()
    ssb = (mean_c.sub(grand) ** 2).mul(cnt, axis=0).sum(); sst = ((d[SAP_FEATS] - grand) ** 2).sum()
    ETA = (ssb / sst).rename('eta2_coin_train').to_frame(); ETA['price_level_feature'] = ETA.index.isin(PRICE_LEVEL); display(ETA.sort_values('eta2_coin_train', ascending=False))
    # nearest-centroid coin identification: fit on train, evaluate on VALIDATION rows of coins present in train
    big = cnt[cnt >= CFG['min_coin_rows_probe']].index; cen = mean_c.loc[big]
    v = pd.DataFrame(SAP_X['val'], columns=SAP_FEATS); v['coin'] = G.loc[G.split == 'val', 'coin'].to_numpy(); v = v[v.coin.isin(big)]
    if len(v) > CFG['probe_eval_rows']: v = v.sample(CFG['probe_eval_rows'], random_state=SEED)
    def centroid_acc(cols):
        C = cen[cols].to_numpy(float); X = v[cols].to_numpy(float); best = np.empty(len(X), dtype=int)
        for i in range(0, len(X), 2000):
            dd = ((X[i:i + 2000, None, :] - C[None, :, :]) ** 2).sum(-1); best[i:i + 2000] = dd.argmin(1)
        return float((cen.index.to_numpy()[best] == v.coin.to_numpy()).mean())
    maj = float(v.coin.value_counts(normalize=True).iloc[0]) if len(v) else np.nan
    non_price = [f for f in SAP_FEATS if f not in PRICE_LEVEL]
    PROBE = {'eval_rows': int(len(v)), 'coins_in_probe': int(len(big)), 'majority_baseline': maj,
             'acc_price_level_only': centroid_acc(PRICE_LEVEL) if PRICE_LEVEL else None, 'acc_non_price_only': centroid_acc(non_price) if non_price else None, 'acc_all_features': centroid_acc(SAP_FEATS)}
    print('Nearest-centroid coin identification on validation rows (train-fitted):'); print(json.dumps(PROBE, indent=1))

# ---------------- feature configurations (rule fixed in advance; train-only evidence) ----------------
cfg_A = list(SAP_FEATS)
if PRIMARY_COIN:
    dropB = [f for f in PRICE_LEVEL if float(ETA.loc[f, 'eta2_coin_train']) >= CFG['eta2_drop']]; evidence_B = 'train eta^2(coin) >= %.2f AND absolute price-level feature' % CFG['eta2_drop']
else:
    dropB = list(PRICE_LEVEL); evidence_B = 'rationale only (absolute price level) - NOT TESTED because no coin identifier exists in the source'
cfg_B = [f for f in SAP_FEATS if f not in dropB]
assert [f for f in cfg_A if f in cfg_B] == cfg_B and set(cfg_B) <= set(cfg_A)                    # B is an order-preserving subset of A
FEATURE_CONFIGS = {'A_original_verified': {'columns': cfg_A, 'n': len(cfg_A), 'rule': 'all 15 frozen columns, frozen order'},
                   'B_price_identity_reduced': {'columns': cfg_B, 'n': len(cfg_B), 'dropped': dropB, 'rule': evidence_B,
                                                'eta2_train': ETA.loc[PRICE_LEVEL, 'eta2_coin_train'].to_dict() if ETA is not None else None},
                   'selection_policy': 'Choose between A and B using VALIDATION (and coin-grouped CV on train) only. The test partition must never inform feature selection. No scaling refit is needed: scaling is per column, so column subsets of the frozen scaled parquets are valid.'}
print(json.dumps(FEATURE_CONFIGS['B_price_identity_reduced'], indent=1, default=jsonable))

,eta2_coin_train,price_level_feature
hour_sin,0.0109974,False
hour_cos,0.00962229,False
log_std_price,0.00452787,True
std_price,0.00377194,True
minute_sin,0.000981593,False
minute_cos,0.000822899,False
avg_rush_order,0.000457438,False
avg_price,0.000452328,True
avg_price_max,0.000408563,True
log_avg_volume,0.000193406,False


Nearest-centroid coin identification on validation rows (train-fitted):
{
 "eval_rows": 20000,
 "coins_in_probe": 71,
 "majority_baseline": 0.10435,
 "acc_price_level_only": 0.01505,
 "acc_non_price_only": 0.02005,
 "acc_all_features": 0.0066
}
{
 "columns": [
  "std_rush_order",
  "avg_rush_order",
  "std_trades",
  "std_volume",
  "avg_volume",
  "std_price",
  "avg_price",
  "avg_price_max",
  "hour_sin",
  "hour_cos",
  "minute_sin",
  "minute_cos",
  "rush_volume_interaction",
  "log_avg_volume",
  "log_std_price"
 ],
 "n": 15,
 "dropped": [],
 "rule": "train eta^2(coin) >= 0.25 AND absolute price-level feature",
 "eta2_train": {
  "std_price": 0.00377194380289774,
  "avg_price": 0.00045232813317864904,
  "avg_price_max": 0.0004085633127399745,
  "log_std_price": 0.004527867501228618
 }
}


In [ ]:
# ---------------- coin-grouped folds INSIDE train (union-find over coin<->pump_index so folds are coin- AND event-disjoint) ----------------
FOLDS = None; FOLD_INFO = {'available': False}
if PRIMARY_COIN:
    T = G[G.split == 'train'][['row_id', 'coin', 'pump_index', 'gt']].copy(); T['coin'] = T.coin.fillna('__NA__')
    parent = {c: c for c in T.coin.unique()}
    def find(a):
        while parent[a] != a: parent[a] = parent[parent[a]]; a = parent[a]
        return a
    for _, cs in T[T.pump_index >= 0].groupby('pump_index')['coin'].unique().items():
        for c in cs[1:]: parent[find(c)] = find(cs[0])
    T['unit'] = T.coin.map(find); U = T.groupby('unit').agg(rows=('gt', 'size'), pos=('gt', 'sum')).reset_index().sort_values(['pos', 'rows', 'unit'], ascending=[False, False, True])
    k = int(min(CFG['coin_folds'], max(2, (U.pos > 0).sum()))); loads = np.zeros((k, 2)); assign = {}
    for u in U.itertuples():
        f = int(np.lexsort((loads[:, 1], loads[:, 0]))[0]); assign[u.unit] = f; loads[f] += (u.pos, u.rows)
    T['fold'] = T.unit.map(assign).astype(np.int8); FOLDS = T[['row_id', 'coin', 'fold']].copy()
    fs = T.groupby('fold').agg(rows=('gt', 'size'), positives=('gt', 'sum'), coins=('coin', 'nunique'), event_groups=('pump_index', lambda s: s[s >= 0].nunique())); display(fs)
    assert T.groupby('coin')['fold'].nunique().max() == 1, 'a coin appears in >1 fold'
    assert T[T.pump_index >= 0].groupby('pump_index')['fold'].nunique().max() == 1, 'an event group appears in >1 fold'
    FOLD_INFO = {'available': True, 'k': k, 'fold_summary': fs.reset_index().to_dict(orient='records'), 'units_merged_by_shared_pump_index': int(len(parent) - T.unit.nunique()),
                 'limitation': 'folds mix time (a held-out coin may be earlier than training coins); they estimate coin generalisation for model selection only and complement, not replace, the chronological benchmark. Simultaneous pumps of different coins (e.g. announced by the same channel) cannot be identified from the saved fields.'}
    # exact-vector overlap between coin-folds (leakage screen)
    Hh = G.loc[G.split == 'train', 'h'].to_numpy(); fo = T.fold.to_numpy(); hf = pd.DataFrame({'h': Hh, 'f': fo}).groupby('h')['f'].nunique()
    FOLD_INFO['train_vectors_shared_across_folds'] = int((hf > 1).sum()); print('train vectors present in >1 coin-fold:', FOLD_INFO['train_vectors_shared_across_folds'], '(identical inactive/imputed background vectors; evaluation protocol: report fold metrics with and without them)')

,rows,positives,coins,event_groups
fold,,,,
0,101669,44,14,44
1,115161,44,14,44
2,143578,44,15,47
3,126085,44,14,46
4,114334,44,14,44


train vectors present in >1 coin-fold: 44384 (identical inactive/imputed background vectors; evaluation protocol: report fold metrics with and without them)


## D · Class imbalance: weights from TRAIN labels, rare-event metric specification, tested metrics module
No resampling is applied anywhere (validation/test prevalence is preserved). SMOTE/over-sampling are **not** used by default: synthetic minority points interpolate across coins, times and event groups, breaking the chronological/event-disjoint structure that makes the benchmark meaningful, and they invite leakage into any CV that is not group-aware.

In [ ]:
EVAL_UTILS_SRC = r"""\"\"\"Rare-event evaluation utilities for TradeSense Layer 2 (source-specific XGBoost benchmarks).\"\"\"
import numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score

def rare_event_metrics(y_true, y_score, threshold, groups=None):
    \"\"\"Metrics at ONE threshold. groups: array aligned with rows; values < 0 or NaN mean 'no event group'.
    Event-level recall = share of positive event groups with >= 1 POSITIVE row predicted positive.\"\"\"
    y = np.asarray(y_true).astype(int); s = np.asarray(y_score, float); pred = (s >= threshold).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum()); fp = int(((pred == 1) & (y == 0)).sum()); fn = int(((pred == 0) & (y == 1)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
    nan = float('nan'); prec = tp / (tp + fp) if tp + fp else nan; rec = tp / (tp + fn) if tp + fn else nan
    f1 = 2 * prec * rec / (prec + rec) if (tp + fp) and (tp + fn) and (prec + rec) > 0 else nan
    both = 0 < y.sum() < len(y)
    out = {'n': int(len(y)), 'positives': int(y.sum()), 'prevalence': float(y.mean()), 'threshold': float(threshold), 'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
           'precision': prec, 'recall': rec, 'f1': f1, 'pr_auc': float(average_precision_score(y, s)) if both else nan, 'roc_auc': float(roc_auc_score(y, s)) if both else nan,
           'false_positives_per_1000_observations': 1000.0 * fp / len(y), 'confusion_matrix': [[tn, fp], [fn, tp]]}
    if groups is not None:
        g = np.asarray(groups, float); pos_groups = np.unique(g[(y == 1) & (g >= 0)])
        det = [bool(((g == k) & (y == 1) & (pred == 1)).any()) for k in pos_groups]
        out.update({'event_groups_positive': int(len(pos_groups)), 'event_groups_detected': int(sum(det)), 'event_recall': float(sum(det) / len(det)) if det else nan})
    return out

def choose_threshold_on_validation(y_val, s_val, max_fp_per_1000=None):
    \"\"\"Pick a threshold on VALIDATION data only: max F1, or max recall subject to a false-positive budget per 1,000 observations.\"\"\"
    y = np.asarray(y_val).astype(int); s = np.asarray(s_val, float); cand = np.unique(np.quantile(s, np.linspace(0, 1, 2001))); best, bt = -1.0, float(cand[-1])
    for t in cand:
        m = rare_event_metrics(y, s, t)
        if max_fp_per_1000 is not None:
            if m['false_positives_per_1000_observations'] <= max_fp_per_1000 and (m['recall'] if m['recall'] == m['recall'] else -1) > best: best, bt = m['recall'], float(t)
        elif m['f1'] == m['f1'] and m['f1'] > best: best, bt = m['f1'], float(t)
    return bt
"""
EVAL_UTILS_SRC = EVAL_UTILS_SRC.replace('\\"', '"')
ns_eval = {}; exec(EVAL_UTILS_SRC, ns_eval); rare_event_metrics = ns_eval['rare_event_metrics']; choose_threshold_on_validation = ns_eval['choose_threshold_on_validation']

def class_weights(y):
    y = np.asarray(y).astype(int); pos, neg = int(y.sum()), int((1 - y).sum())
    return {'positives': pos, 'negatives': neg, 'scale_pos_weight_full': neg / pos if pos else None, 'scale_pos_weight_sqrt': math.sqrt(neg / pos) if pos else None, 'scale_pos_weight_none': 1.0}
SAP_W = {'primary_unfiltered_train': class_weights(SAP_PQ['train'][SAP_T]), 'filtered_train_option': class_weights(y_after) if FILTER_JUSTIFIED else None,
         'frozen_metadata_scale_pos_weight': float(SAP_M['scale_pos_weight']), 'policy': 'weights computed from TRAIN labels only; candidates {full, sqrt, none} to be chosen on validation PR-AUC; no resampling'}
assert abs(SAP_W['primary_unfiltered_train']['scale_pos_weight_full'] - SAP_W['frozen_metadata_scale_pos_weight']) < 1e-6 * SAP_W['frozen_metadata_scale_pos_weight']
if FOLDS is not None:
    ytr = SAP_PQ['train'][SAP_T].to_numpy(); SAP_W['coin_fold_train_weights'] = {int(f): class_weights(ytr[~(FOLDS.fold.to_numpy() == f)]) for f in sorted(FOLDS.fold.unique())}
print(json.dumps(SAP_W, indent=1, default=jsonable)[:1800])
METRIC_SPEC = {'metrics': ['precision', 'recall', 'f1', 'pr_auc', 'roc_auc', 'confusion_matrix', 'false_positives_per_1000_observations', 'event_recall (Sapienza: pump_index groups)'],
               'primary_metric_for_model_selection': 'pr_auc on VALIDATION (never test)', 'threshold_policy': 'choose threshold on VALIDATION only (max F1 or FP-budget); freeze; apply once to test',
               'false_positives_per_1000_observations': '1000 * FP / (all rows evaluated)', 'event_recall': 'share of positive event groups with >= 1 positive row predicted positive',
               'prevalence': 'preserve original positive prevalence; no resampling of validation/test', 'accuracy': 'must not be reported alone',
               'report_views': ['full split (primary)', 'coin-unseen subset (coin generalisation)', 'rows whose exact vector is NOT in train (novel-vector view)'],
               'implementation': 'evaluation/layer2_eval_utils.py (unit-tested during validation)'}

{
 "primary_unfiltered_train": {
  "positives": 220,
  "negatives": 600607,
  "scale_pos_weight_full": 2730.0318181818184,
  "scale_pos_weight_sqrt": 52.24970639325946,
  "scale_pos_weight_none": 1.0
 },
 "filtered_train_option": null,
 "frozen_metadata_scale_pos_weight": 2730.0318181818184,
 "policy": "weights computed from TRAIN labels only; candidates {full, sqrt, none} to be chosen on validation PR-AUC; no resampling",
 "coin_fold_train_weights": {
  "0": {
   "positives": 176,
   "negatives": 498982,
   "scale_pos_weight_full": 2835.125,
   "scale_pos_weight_sqrt": 53.24589186031163,
   "scale_pos_weight_none": 1.0
  },
  "1": {
   "positives": 176,
   "negatives": 485490,
   "scale_pos_weight_full": 2758.465909090909,
   "scale_pos_weight_sqrt": 52.52109965614685,
   "scale_pos_weight_none": 1.0
  },
  "2": {
   "positives": 176,
   "negatives": 457073,
   "scale_pos_weight_full": 2597.005681818182,
   "scale_pos_weight_sqrt": 50.96082497191526,
   "scale_pos_weight_none": 1.0
  

## E · Mt.Gox: label provenance, chronology, entity identifiers, why identical vectors carry different labels
The raw source is read with **all** columns (numeric extras as float32, text as category; the frozen feature columns and label as float64 so the replay stays bit-exact). Everything below is *evidence gathering from the source as it exists*: no timestamp, entity ID or label is invented, no conflicting cluster is relabelled or removed.

In [ ]:
GOX_M = joblib.load(L2_ROOT / 'mtgox_metadata.joblib'); GOX_FEATS = list(GOX_M['features']); GOX_T = 'wash'
GOX_PQ = {sp: pd.read_parquet(L2_ROOT / f'mtgox_{sp}.parquet') for sp in SPL}
assert list(GOX_PQ['train'].columns) == GOX_FEATS + [GOX_T]
GOX_N = {sp: len(v) for sp, v in GOX_PQ.items()}; N_GOX = sum(GOX_N.values()); a_idx, b_idx = int(N_GOX * .70), int(N_GOX * .85)
GOX_BOUNDS = {'train': (0, a_idx), 'val': (a_idx, b_idx), 'test': (b_idx, N_GOX)}
assert {sp: hi - lo for sp, (lo, hi) in GOX_BOUNDS.items()} == GOX_N, 'frozen row counts != 70/15/15 row-index split of the source'
ACT_RE  = re.compile(r'count|_24h|_7d|_30d|rate|ratio|avg|mean|std|sum|num|total|volume|btc|usd|price|dev|amount|size|deviation', re.I)
TIME_RE = re.compile(r'time|date|stamp|^ts$|epoch|^year$|^month$|^day$|^hour$', re.I)
PART_RE = re.compile(r'(^|_)(user|trader|account|wallet|address|participant|counterpart\w*|buyer|seller|party)(_id)?$', re.I)      # names that denote a PARTICIPANT
GEN_ID_RE = re.compile(r'(^|_)(id|uid|uuid|guid|hash)$', re.I)                                                                     # generic identifier: transaction OR participant (undeterminable from the name)
LAB_RE  = re.compile(r'wash|label|target|flag|class|fraud|suspic', re.I)
hdr = list(pd.read_csv(GOX_FILE, nrows=0).columns); probe = pd.read_csv(GOX_FILE, nrows=50_000)
dmap = {c: ('float64' if (c in GOX_BASE + [GOX_T] or TIME_RE.search(c) or ((PART_RE.search(c) or GEN_ID_RE.search(c)) and not ACT_RE.search(c))) and pd.api.types.is_numeric_dtype(probe[c]) else ('float32' if pd.api.types.is_numeric_dtype(probe[c]) else 'category')) for c in hdr}
GX = pd.read_csv(GOX_FILE, dtype=dmap); del probe
assert len(GX) == N_GOX, f'source rows {len(GX)} != frozen total {N_GOX}'
print(f'Mt.Gox source: {len(GX):,} rows x {len(hdr)} columns | memory {GX.memory_usage(deep=True).sum()/1e6:.0f} MB'); print('columns:', hdr)

EXTRA_G = [c for c in hdr if c not in GOX_BASE + [GOX_T]]
roles = pd.DataFrame({'column': hdr, 'in_features_or_target': [c in GOX_BASE + [GOX_T] for c in hdr], 'dtype': [str(GX[c].dtype) for c in hdr],
                      'n_unique': [GX[c].nunique() for c in hdr], 'time_like_name': [bool(TIME_RE.search(c)) for c in hdr],
                      'participant_like_name': [bool(PART_RE.search(c)) and not ACT_RE.search(c) for c in hdr], 'generic_id_name': [bool(GEN_ID_RE.search(c)) and not ACT_RE.search(c) for c in hdr], 'activity_like_name': [bool(ACT_RE.search(c)) for c in hdr],
                      'label_like_name': [bool(LAB_RE.search(c)) and c != GOX_T for c in hdr]}).set_index('column')
display(roles)
print('NOTE: columns such as *_trade_count / *_24h / *_7d are ACTIVITY COUNTS, never participant identifiers.')
GOX_DOCS_DIRS = [p for p in BASE.iterdir() if p.is_dir() and re.search(r'gox', p.name, re.I)]
DOC_FILES = [p for d in GOX_DOCS_DIRS for p in scan_docs(d)]
GOX_DOC_HITS = {}
for p in DOC_FILES:
    lines = p.read_text(encoding='utf-8', errors='replace').splitlines(); hits = [l.strip()[:220] for l in lines if re.search(r'wash|label|self|round|trader|user|time|order|sybil|heuristic|ground', l, re.I)][:25]
    GOX_DOC_HITS[str(p.relative_to(BASE))] = hits; print(f'\n--- {p.relative_to(BASE)} ({len(lines)} lines) ---'); [print('  ', h) for h in hits]
print('\nMt.Gox documentation files found near the source:', [str(p.relative_to(BASE)) for p in DOC_FILES] or 'NONE')

Mt.Gox source: 5,537,252 rows x 17 columns | memory 509 MB
columns: ['bitcoins', 'round_level', 'cumu_wash_percent_opt', 'buyer_24h_trade_count', 'buyer_7d_trade_count', 'seller_24h_trade_count', 'seller_7d_trade_count', 'buyer_24h_btc_sum', 'buyer_7d_btc_sum', 'seller_24h_btc_sum', 'seller_7d_btc_sum', 'buyer_all_btc_sum', 'seller_all_btc_sum', 'price_deviation', 'time_since_last_trade', 'hours', 'wash']


,in_features_or_target,dtype,n_unique,time_like_name,participant_like_name,generic_id_name,activity_like_name,label_like_name
column,,,,,,,,
bitcoins,True,float64,2094362,False,False,False,False,False
round_level,False,float32,1676983,False,False,False,False,False
cumu_wash_percent_opt,False,float32,2201095,False,False,False,False,True
buyer_24h_trade_count,True,float64,5756,False,False,False,True,False
buyer_7d_trade_count,False,float32,6762,False,False,False,True,False
seller_24h_trade_count,True,float64,5849,False,False,False,True,False
seller_7d_trade_count,False,float32,6891,False,False,False,True,False
buyer_24h_btc_sum,False,float32,481008,False,False,False,True,False
buyer_7d_btc_sum,False,float32,517455,False,False,False,True,False


NOTE: columns such as *_trade_count / *_24h / *_7d are ACTIVITY COUNTS, never participant identifiers.

Mt.Gox documentation files found near the source: NONE


In [ ]:
# ---------------- chronology evidence: is there a reliable time column, and is the file order chronological? ----------------
def assess_time_col(s, n_probe=100_000):
    n = len(s); idx = np.linspace(0, n - 1, min(n, n_probe)).astype(int); v = s.iloc[idx]
    if str(s.dtype) in ('category', 'object', 'string'):
        t = pd.to_datetime(v.astype(str), errors='coerce', utc=True); arr = t.dropna().astype('int64').to_numpy(); ok = float(t.notna().mean())
    else:
        x = pd.to_numeric(v, errors='coerce'); ok = float(x.notna().mean()); arr = x.dropna().to_numpy(float)
        med = np.median(arr) if len(arr) else 0.0
        t = pd.to_datetime(x, unit='s' if med < 1e11 else ('ms' if med < 1e14 else 'us'), errors='coerce', utc=True) if med > 3e8 else x
    mono = float((np.diff(arr) >= 0).mean()) if len(arr) > 1 else float('nan'); nun = int(pd.Series(arr).nunique())
    tt = t.dropna()
    return {'parse_ok_share': ok, 'monotonic_non_decreasing_share_sampled': mono, 'n_unique_in_sample': nun, 'min': str(tt.min()) if len(tt) else None, 'max': str(tt.max()) if len(tt) else None,
            'sampled_rows': int(len(idx)), 'reliable_chronology': bool(ok >= 0.99 and mono >= CFG['mono_fraction'] and nun >= CFG['time_min_unique'])}
TIME_COLS = [c for c in EXTRA_G if TIME_RE.search(c)]
TIME_EVID = {c: assess_time_col(GX[c]) for c in TIME_COLS}
print('time-like columns:', TIME_COLS or 'NONE'); [print(c, json.dumps(v)) for c, v in TIME_EVID.items()]
CHRONO_SUPPORTED = any(v['reliable_chronology'] for v in TIME_EVID.values())
TIME_VALID_BUT_UNSORTED = [c for c, v in TIME_EVID.items() if v['parse_ok_share'] >= 0.99 and v['n_unique_in_sample'] >= CFG['time_min_unique'] and not v['reliable_chronology']]
ENTITY_CANDIDATES = [c for c in roles.index if roles.loc[c, 'participant_like_name'] and c not in GOX_BASE]
GENERIC_ID_COLS = [c for c in roles.index if roles.loc[c, 'generic_id_name'] and c not in GOX_BASE]
ENT_EVID = {c: {'kind': 'participant-like name' if c in ENTITY_CANDIDATES else 'generic id (transaction vs participant undeterminable)', 'n_unique': int(GX[c].nunique()), 'share_top_value': float(GX[c].value_counts(normalize=True).iloc[0]), 'rows': len(GX)} for c in ENTITY_CANDIDATES + GENERIC_ID_COLS}
print('participant-like columns:', ENTITY_CANDIDATES or 'NONE', '| generic ID columns (not evidence of entity identity):', GENERIC_ID_COLS or 'NONE'); print(json.dumps(ENT_EVID, indent=1))

time-like columns: ['time_since_last_trade']
time_since_last_trade {"parse_ok_share": 1.0, "monotonic_non_decreasing_share_sampled": 0.6649566495664957, "n_unique_in_sample": 483, "min": "0.0", "max": "16666.0", "sampled_rows": 100000, "reliable_chronology": false}
participant-like columns: NONE | generic ID columns (not evidence of entity identity): NONE
{}


In [ ]:
# ---------------- replay: row-index split + frozen transform reproduces the saved parquets bit-for-bit ----------------
def gox_features(d):
    x = d[GOX_BASE].apply(pd.to_numeric, errors='coerce').copy()
    x['trade_count_ratio'] = x['buyer_24h_trade_count'] / (x['seller_24h_trade_count'] + 1)
    x['log_bitcoins'] = np.log1p(x['bitcoins'].clip(lower=0)); x['abs_price_deviation'] = x['price_deviation'].abs()
    return x.replace([np.inf, -np.inf], np.nan)
GOX_ALIGN, xs_raw = {}, {}
for sp, (lo, hi) in GOX_BOUNDS.items():
    part = GX.iloc[lo:hi]; x = gox_features(part); xs_raw[sp] = x
    arr = GOX_M['scaler'].transform(GOX_M['imputer'].transform(x)).astype(np.float32); saved = GOX_PQ[sp][GOX_FEATS].to_numpy(np.float32)
    GOX_ALIGN[sp] = {'rows': hi - lo, 'features_bit_identical': bool(np.array_equal(arr, saved)), 'labels_identical': bool(np.array_equal(part[GOX_T].to_numpy().astype(np.int8), GOX_PQ[sp][GOX_T].to_numpy().astype(np.int8)))}
    del arr, saved
display(pd.DataFrame(GOX_ALIGN).T)
assert all(a['features_bit_identical'] and a['labels_identical'] for a in GOX_ALIGN.values()), f'Mt.Gox replay FAILED: {GOX_ALIGN}'
g_train = max_rel_diff((GOX_M['imputer'].statistics_, GOX_M['scaler'].center_, GOX_M['scaler'].scale_), fit_params_numpy(xs_raw['train']))
g_all = max_rel_diff((GOX_M['imputer'].statistics_, GOX_M['scaler'].center_, GOX_M['scaler'].scale_), fit_params_numpy(pd.concat(xs_raw.values())))
assert g_train < 1e-6, f'Mt.Gox saved preprocessing != train-only recomputation ({g_train:.2e})'
print(f'Mt.Gox replay bit-exact; train-only fit proven (rel diff {g_train:.1e} vs all-partition {g_all:.1e})'); del xs_raw

# ---------------- repeated-vector structure over the FROZEN features (global row order == source order) ----------------
GH = np.concatenate([hash_rows(GOX_PQ[sp], GOX_FEATS) for sp in SPL]); GY = np.concatenate([GOX_PQ[sp][GOX_T].to_numpy().astype(np.int8) for sp in SPL])
GS = np.concatenate([np.full(GOX_N[sp], i, dtype=np.int8) for i, sp in enumerate(SPL)])
cl = pd.DataFrame({'h': GH, 'y': GY, 's': GS}); cg = cl.groupby('h')
cl['n'] = cg['y'].transform('size'); cl['pos'] = cg['y'].transform('sum'); cl['mixed'] = (cl.pos > 0) & (cl.pos < cl.n)
tr_stats = cl[cl.s == 0].groupby('h')['y'].agg(['size', 'sum']); tr_stats.columns = ['n_tr', 'pos_tr']
cl = cl.join(tr_stats, on='h'); cl['n_tr'] = cl.n_tr.fillna(0).astype(np.int32); cl['pos_tr'] = cl.pos_tr.fillna(0)
cl['train_cluster_mixed'] = (cl.pos_tr > 0) & (cl.pos_tr < cl.n_tr); cl['vec_seen_in_train'] = (cl.n_tr > 0) & (cl.s > 0)
rep_g = cl.n >= 2
floor = float((np.minimum(cl.pos, cl.n - cl.pos) / cl.n)[rep_g].mean())                    # irreducible error among repeated vectors (memorisation lower bound)
GOX_REPEAT = {'rows': int(len(cl)), 'unique_vectors': int(cl.h.nunique()), 'share_rows_in_repeated_vectors': float(rep_g.mean()), 'mixed_label_clusters': int(cl.loc[cl.mixed].h.nunique()),
              'share_of_repeated_rows_in_mixed_clusters': float(cl.mixed[rep_g].mean()), 'bayes_error_floor_on_repeated_rows': floor,
              'eval_rows_vector_in_train': {sp: float(cl.loc[cl.s == i, 'vec_seen_in_train'].mean()) for i, sp in ((1, 'val'), (2, 'test'))},
              'eval_positives_vector_in_train': {sp: int(((cl.s == i) & cl.vec_seen_in_train & (cl.y == 1)).sum()) for i, sp in ((1, 'val'), (2, 'test'))}}
print(json.dumps(GOX_REPEAT, indent=1))

,rows,features_bit_identical,labels_identical
train,3876076,True,True
val,830588,True,True
test,830588,True,True


Mt.Gox replay bit-exact; train-only fit proven (rel diff 0.0e+00 vs all-partition 6.0e-01)
{
 "rows": 5537252,
 "unique_vectors": 4843433,
 "share_rows_in_repeated_vectors": 0.1541723222999423,
 "mixed_label_clusters": 11923,
 "share_of_repeated_rows_in_mixed_clusters": 0.08623846333158017,
 "bayes_error_floor_on_repeated_rows": 0.022133301159318778,
 "eval_rows_vector_in_train": {
  "val": 0.026789455181148777,
  "test": 0.02670156563783729
 },
 "eval_positives_vector_in_train": {
  "val": 1251,
  "test": 1144
 }
}


In [ ]:
# ---------------- label-derivation evidence ----------------
def auc_rank(x, y):
    m = np.isfinite(x); x, y = x[m], y[m]; n1 = int(y.sum()); n0 = len(y) - n1
    if n1 == 0 or n0 == 0 or np.nanstd(x) == 0: return float('nan')
    r = stats.rankdata(x); return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

samp = np.random.default_rng(SEED).choice(N_GOX, size=min(N_GOX, CFG['sample_auc_rows']), replace=False); samp.sort()
ys = GY[samp]; HID = []
for c in EXTRA_G:
    if str(GX[c].dtype) == 'category':
        t = pd.DataFrame({'v': GX[c].to_numpy()[samp], 'y': ys}).groupby('v', observed=True)['y'].agg(['size', 'mean']); acc = float((np.where(t['mean'] > .5, t['mean'], 1 - t['mean']) * t['size']).sum() / t['size'].sum())
        HID.append({'column': c, 'kind': 'categorical', 'auc': np.nan, 'auc_dev': np.nan, 'max_likelihood_accuracy': acc, 'n_levels': len(t)})
    else:
        a = auc_rank(GX[c].to_numpy(float)[samp], ys); HID.append({'column': c, 'kind': 'numeric', 'auc': a, 'auc_dev': abs(a - .5) if a == a else np.nan, 'max_likelihood_accuracy': np.nan, 'n_levels': int(GX[c].nunique())})
HID = pd.DataFrame(HID).set_index('column') if HID else pd.DataFrame(columns=['kind', 'auc', 'auc_dev', 'max_likelihood_accuracy', 'n_levels'])
print('Association of each NON-feature source column with the label (400k-row deterministic sample):'); display(HID.sort_values('auc_dev', ascending=False))
_xs = gox_features(GX.iloc[samp]); FEAT_AUC = {f: auc_rank(_xs[f].to_numpy(float), ys) for f in GOX_FEATS}; del _xs
print('single-feature AUCs of the MODEL features (derived features included):', {k: round(v, 3) for k, v in FEAT_AUC.items()})
HID_DET = [c for c in HID.index if (HID.loc[c, 'kind'] == 'numeric' and HID.loc[c, 'auc_dev'] >= CFG['auc_deterministic_dev']) or (HID.loc[c, 'kind'] == 'categorical' and HID.loc[c, 'max_likelihood_accuracy'] >= 0.99)]
FEAT_DET = [f for f, a in FEAT_AUC.items() if a == a and abs(a - .5) >= CFG['auc_deterministic_dev']]
print('near-deterministic hidden predictors of the label:', HID_DET or 'none', '| near-deterministic MODEL features:', FEAT_DET or 'none')

# ---- why do identical vectors carry different labels? within-cluster association with hidden columns ----
mixed_rows = np.flatnonzero(cl.mixed.to_numpy())
CONFLICT_EXPL = {}
if len(mixed_rows):
    sel = np.random.default_rng(SEED).choice(mixed_rows, size=min(len(mixed_rows), 200_000), replace=False); sel.sort()
    hh, yy = GH[sel], GY[sel]
    for c in EXTRA_G:
        if str(GX[c].dtype) == 'category': continue
        v = GX[c].to_numpy(float)[sel]; vc = v - pd.Series(v).groupby(hh).transform('mean').to_numpy()
        a = auc_rank(vc, yy); CONFLICT_EXPL[c] = {'within_cluster_auc': a, 'dev': abs(a - .5) if a == a else None}
    print('Within-conflict-cluster AUC of hidden numeric columns (cluster mean removed):'); display(pd.DataFrame(CONFLICT_EXPL).T)
EXPLAINERS = [c for c, v in CONFLICT_EXPL.items() if v['dev'] is not None and v['dev'] >= CFG['auc_explains_conflict_dev']]
print('hidden columns that separate labels WITHIN identical-feature clusters (information-loss evidence):', EXPLAINERS or 'none')
prate = [float(b.mean()) for b in np.array_split(GY, CFG['mtgox_blocks'])]; print('positive rate across 10 sequential blocks:', [round(p, 3) for p in prate])

Association of each NON-feature source column with the label (400k-row deterministic sample):


,kind,auc,auc_dev,max_likelihood_accuracy,n_levels
column,,,,,
buyer_7d_trade_count,numeric,0.743583,0.243583,NaN,6762
buyer_all_btc_sum,numeric,0.729961,0.229961,NaN,5049870
seller_7d_trade_count,numeric,0.704374,0.204374,NaN,6891
seller_all_btc_sum,numeric,0.697382,0.197382,NaN,4989148
buyer_7d_btc_sum,numeric,0.68878,0.18878,NaN,517455
seller_7d_btc_sum,numeric,0.661383,0.161383,NaN,463123
buyer_24h_btc_sum,numeric,0.622405,0.122405,NaN,481008
seller_24h_btc_sum,numeric,0.608878,0.108878,NaN,418774
cumu_wash_percent_opt,numeric,0.40577,0.09423,NaN,2201095


single-feature AUCs of the MODEL features (derived features included): {'bitcoins': 0.434, 'price_deviation': 0.494, 'buyer_24h_trade_count': 0.698, 'seller_24h_trade_count': 0.668, 'trade_count_ratio': 0.523, 'log_bitcoins': 0.434, 'abs_price_deviation': 0.529}
near-deterministic hidden predictors of the label: none | near-deterministic MODEL features: none
Within-conflict-cluster AUC of hidden numeric columns (cluster mean removed):


,within_cluster_auc,dev
round_level,0.499767,0.000232632
cumu_wash_percent_opt,0.398639,0.101361
buyer_7d_trade_count,0.738064,0.238064
seller_7d_trade_count,0.684984,0.184984
buyer_24h_btc_sum,0.523428,0.0234283
buyer_7d_btc_sum,0.703543,0.203543
seller_24h_btc_sum,0.548614,0.0486135
seller_7d_btc_sum,0.668016,0.168016
buyer_all_btc_sum,0.743781,0.243781
seller_all_btc_sum,0.682979,0.182979


hidden columns that separate labels WITHIN identical-feature clusters (information-loss evidence): ['buyer_7d_trade_count', 'buyer_7d_btc_sum', 'buyer_all_btc_sum']
positive rate across 10 sequential blocks: [0.348, 0.414, 0.494, 0.43, 0.396, 0.333, 0.34, 0.245, 0.223, 0.214]


In [ ]:
# ---------------- classification rule (printed, evidence-driven; never forced) ----------------
def classify_mtgox():
    reasons_C, reasons_B, reasons_A = [], [], []
    py = float(GY.mean())
    if py < 0.005 or py > 0.995: reasons_C.append(f'label is almost constant (positive rate {py:.3f})')
    if FEAT_DET: reasons_C.append(f'model feature(s) alone near-deterministically predict the label: {FEAT_DET}')
    if floor >= CFG['gox_conflict_floor_unsuitable'] and GOX_REPEAT['share_rows_in_repeated_vectors'] >= CFG['gox_repeat_share_unsuitable']:
        reasons_C.append(f'majority of rows are repeated vectors and their irreducible error is {floor:.1%}')
    if HID_DET: reasons_B.append(f'label appears derived from hidden source column(s) {HID_DET}: a model on the frozen features would learn a proxy of that rule, not wash behaviour')
    if not DOC_FILES: reasons_B.append('no documentation of the wash label definition (directly observed vs estimated vs derived) was found next to the source')
    if not CHRONO_SUPPORTED: reasons_B.append('no reliable time column supports chronology of the row-index split' + (f'; time-like columns exist but file order is not chronological: {TIME_VALID_BUT_UNSORTED}' if TIME_VALID_BUT_UNSORTED else ''))
    if not ENTITY_CANDIDATES: reasons_B.append('no participant/entity identifier exists, so entity independence of the split cannot be established (activity counts are not IDs)' + (f'; generic ID column(s) {GENERIC_ID_COLS} exist but cannot be shown to identify participants (they may be transaction IDs)' if GENERIC_ID_COLS else ''))
    if floor > CFG['gox_conflict_floor_qualified']: reasons_B.append(f'identical vectors with conflicting labels: irreducible error on repeated rows {floor:.1%}')
    if EXPLAINERS: reasons_B.append(f'conflicts are partly explained by fields not in the model features ({EXPLAINERS}) -> information loss')
    elif GOX_REPEAT['mixed_label_clusters'] > 0: reasons_B.append('conflicting labels are NOT explained by any other source column -> label noise or non-identifiable observations (cause unverifiable)')
    drift = max(prate) / max(min(prate), 1e-12)
    if drift > 1.5: reasons_B.append(f'positive rate drifts {min(prate):.2f}->{max(prate):.2f} across sequential blocks (x{drift:.1f}); label prevalence is not stationary')
    if reasons_C: return 'C', reasons_C + reasons_B
    if reasons_B: return 'B', reasons_B
    return 'A', ['documentation present, chronology supported, no derivation proxy, conflicts below threshold']
GOX_CLASS, GOX_REASONS = classify_mtgox()
GOX_CLASS_NAME = {'A': 'A: suitable for a qualified supervised benchmark', 'B': 'B: exploratory benchmark with substantial label/split limitations', 'C': 'C: unsuitable for reliable supervised conclusions'}[GOX_CLASS]
print('MT.GOX CLASSIFICATION ->', GOX_CLASS_NAME); [print('  -', r) for r in GOX_REASONS]
GOX_VERDICT = {'A': 'READY', 'B': 'EXPLORATORY', 'C': 'BLOCKED'}[GOX_CLASS]

# ---------------- improved split ONLY if a valid time column exists but the file order is not chronological ----------------
GOX_IMPROVED = None
if TIME_VALID_BUT_UNSORTED and not CHRONO_SUPPORTED:
    tc = TIME_VALID_BUT_UNSORTED[0]; tv = pd.to_numeric(GX[tc], errors='coerce').to_numpy(float) if str(GX[tc].dtype) != 'category' else pd.to_datetime(GX[tc].astype(str), utc=True, errors='coerce').astype('int64').to_numpy()
    order = np.argsort(tv, kind='stable'); a2, b2 = int(len(order) * .70), int(len(order) * .85)
    spl = np.empty(len(order), dtype='U5'); spl[order[:a2]] = 'train'; spl[order[a2:b2]] = 'val'; spl[order[b2:]] = 'test'
    from sklearn.impute import SimpleImputer; from sklearn.preprocessing import RobustScaler
    x_all = gox_features(GX); imp2 = SimpleImputer(strategy='median', keep_empty_features=True).fit(x_all[spl == 'train']); sc2 = RobustScaler().fit(imp2.transform(x_all[spl == 'train']))   # fitted on the NEW TRAIN ONLY
    GOX_IMPROVED = {'time_column': tc, 'manifest': pd.DataFrame({'source_row_id': np.arange(len(order), dtype=np.int64), 'split': spl}), 'imputer': imp2, 'scaler': sc2,
                    'counts': {s: int((spl == s).sum()) for s in ('train', 'val', 'test')}, 'positive_rate': {s: float(GY[spl == s].mean()) for s in ('train', 'val', 'test')}}
    print('IMPROVED chronological split prepared from', tc, GOX_IMPROVED['counts'], GOX_IMPROVED['positive_rate'])
else: print('No improved split: ' + ('file order already chronological per a reliable time column' if CHRONO_SUPPORTED else 'no valid time column exists (row-index split kept; NOT claimed to be time-safe)'))

MT.GOX CLASSIFICATION -> B: exploratory benchmark with substantial label/split limitations
  - no documentation of the wash label definition (directly observed vs estimated vs derived) was found next to the source
  - no reliable time column supports chronology of the row-index split
  - no participant/entity identifier exists, so entity independence of the split cannot be established (activity counts are not IDs)
  - conflicts are partly explained by fields not in the model features (['buyer_7d_trade_count', 'buyer_7d_btc_sum', 'buyer_all_btc_sum']) -> information loss
  - positive rate drifts 0.21->0.49 across sequential blocks (x2.3); label prevalence is not stationary
No improved split: no valid time column exists (row-index split kept; NOT claimed to be time-safe)


In [ ]:
# ---------------- Mt.Gox row flags, class weights, evaluation protocol ----------------
GOX_FLAGS = {}
for i, sp in enumerate(SPL):
    m = cl.s.to_numpy() == i
    GOX_FLAGS[sp] = pd.DataFrame({'vector_seen_in_train': cl.vec_seen_in_train.to_numpy()[m] if sp != 'train' else np.zeros(m.sum(), bool), 'train_cluster_label_mixed': cl.train_cluster_mixed.to_numpy()[m],
                                  'n_same_vector_in_train': cl.n_tr.to_numpy()[m].astype(np.int32)})
GOX_W = {'train': class_weights(GOX_PQ['train'][GOX_T]), 'frozen_metadata_scale_pos_weight': float(GOX_M['scale_pos_weight']),
         'policy': 'weights from TRAIN labels only; candidates {full, sqrt, none}; chosen on validation PR-AUC; no resampling'}
assert abs(GOX_W['train']['scale_pos_weight_full'] - GOX_W['frozen_metadata_scale_pos_weight']) < 1e-6 * GOX_W['frozen_metadata_scale_pos_weight']
GOX_CHECK_EXCL = {'excluded_rows': 0, 'reason': 'no row is excluded or relabelled: conflicting clusters are retained (information loss / unverifiable label noise, not an established data error); no genuine data error was established from the available source columns.'}
print(json.dumps(GOX_W, indent=1, default=jsonable))

{
 "train": {
  "positives": 1525207,
  "negatives": 2350869,
  "scale_pos_weight_full": 1.5413442240954833,
  "scale_pos_weight_sqrt": 1.2415088497854065,
  "scale_pos_weight_none": 1.0
 },
 "frozen_metadata_scale_pos_weight": 1.5413442240954833,
 "policy": "weights from TRAIN labels only; candidates {full, sqrt, none}; chosen on validation PR-AUC; no resampling"
}


## G · Write the artifacts (staged), manifests and README
Everything is written into the hidden staging directory. Frozen v3 parquets/joblibs are **referenced by absolute path + SHA-256, not copied**. Nothing is published until section H passes.

In [ ]:
t0 = time.time()
STAGE.mkdir(parents=True, exist_ok=False)
SD, GD = STAGE / 'sapienza', STAGE / 'mtgox'
for d in ('metadata', 'model_data', 'preprocessing', 'evaluation', 'reports'): (SD / d).mkdir(parents=True)
for d in ('model_data', 'preprocessing', 'evaluation', 'reports'): (GD / d).mkdir(parents=True)
excl_ids = set(EXCL.row_id.tolist()) if FILTER_JUSTIFIED else set()
fold_of = dict(zip(FOLDS.row_id.tolist(), FOLDS.fold.tolist())) if FOLDS is not None else {}

# ---------------- Sapienza sidecars (row-aligned: sidecar.row_id == positional index in the frozen parquet) ----------------
SAP_SIDECAR_FILES = {}
for sp in SPL:
    g = G[G.split == sp].reset_index(drop=True); part = SAP_PARTS[sp]
    sc = pd.DataFrame({'row_id': g.row_id.to_numpy(np.int32), 'split': sp, 'source_row_id': g.source_row_id.to_numpy(np.int64), 'date': part['date'].to_numpy(),
                       'pump_index': part['pump_index'].to_numpy(float), 'event_group_id': part['pump_index'].where(part['pump_index'] >= 0).astype('Int64').to_numpy(),
                       'gt': g['gt'].to_numpy(np.int8), 'n_missing_features': g.n_missing.to_numpy(np.int8)})
    for c in ID_COLS: sc[c] = part[c].astype('string').to_numpy()
    sc['coin'] = pd.array(g.coin.to_numpy(), dtype='string') if PRIMARY_COIN else pd.array([None] * len(g), dtype='string')
    sc['vec_hash'] = g.h.to_numpy(np.uint64); sc['dup_category'] = g.dup_category.astype('string').to_numpy()
    sc['vec_seen_in_train'] = g.vec_seen_in_train.to_numpy(bool); sc['coin_seen_in_train'] = g.coin_seen_in_train.to_numpy(bool) if PRIMARY_COIN else False
    sc['coin_unseen_eval_subset'] = (~g.coin_seen_in_train.to_numpy(bool)) & (sp != 'train') if PRIMARY_COIN else False
    sc['train_filter_excluded'] = (sc.row_id.isin(excl_ids).to_numpy() if sp == 'train' else False)
    sc['coin_fold'] = pd.array([fold_of.get(int(r), None) for r in sc.row_id], dtype='Int8') if (sp == 'train' and FOLDS is not None) else pd.array([None] * len(sc), dtype='Int8')
    p = SD / 'metadata' / f'sapienza_{sp}_sidecar.parquet'; sc.to_parquet(p, index=False); SAP_SIDECAR_FILES[sp] = p
    print(f'  wrote {p.relative_to(STAGE)}  {sc.shape}  {p.stat().st_size/1e6:.1f} MB')

# ---------------- Sapienza model_data / evaluation / preprocessing / reports ----------------
dump_json(FEATURE_CONFIGS, SD / 'model_data' / 'feature_configs.json')
if FILTER_JUSTIFIED: EXCL[['row_id', 'source_row_id', 'reason']].to_parquet(SD / 'model_data' / 'train_filter_identical_source_records.parquet', index=False)
if FOLDS is not None: FOLDS.to_parquet(SD / 'evaluation' / 'coin_grouped_folds_train.parquet', index=False)
dump_json(SAP_W, SD / 'evaluation' / 'class_weights.json'); dump_json(METRIC_SPEC, SD / 'evaluation' / 'metric_spec.json'); (SD / 'evaluation' / 'layer2_eval_utils.py').write_text(EVAL_UTILS_SRC, encoding='utf-8')
SAP_PREP_REF = {'artifact': str(L2_ROOT / 'sapienza_metadata.joblib'), 'sha256': FROZEN_PIN['sapienza_metadata.joblib']['sha256'], 'imputer': 'sklearn SimpleImputer(median)', 'scaler': 'sklearn RobustScaler',
                'fitted_on': 'TRAIN partition only (independent numpy recomputation from the raw source matches to %.1e; an all-partition fit would differ by %.1e)' % (p_train, p_all),
                'already_applied_to_frozen_parquets': True, 'refit_policy': 'NEVER refit. This directory creates no new imputer/scaler. Column subsets (config B) need no refit because scaling is per column.',
                'imputer_statistics': SAP_IMP.statistics_.tolist(), 'scaler_center': SAP_SC.center_.tolist(), 'scaler_scale': SAP_SC.scale_.tolist(), 'feature_order': SAP_FEATS}
dump_json(SAP_PREP_REF, SD / 'preprocessing' / 'preprocessing_reference.json')
dump_json({**SAP_FACTS, 'primary_coin_column': PRIMARY_COIN, 'identifier_columns_by_name': ID_COLS, 'source_columns': SRC_COLS, 'fields_unavailable': [f for f, ok in [('symbol/coin identifier', PRIMARY_COIN is not None)] if not ok]}, SD / 'reports' / 'alignment_report.json')
dump_json(DUP_DECISION, SD / 'reports' / 'duplicates_report.json')
dump_json({'coin': COIN, 'eta2_train': ETA.reset_index().to_dict(orient='records') if ETA is not None else None, 'identity_probe_validation_rows': PROBE, 'feature_configs': FEATURE_CONFIGS, 'coin_folds': FOLD_INFO}, SD / 'reports' / 'coin_identity_report.json')

# ---------------- Mt.Gox ----------------
for sp in SPL: GOX_FLAGS[sp].to_parquet(GD / 'evaluation' / f'row_flags_{sp}.parquet', index=False)
dump_json(GOX_W, GD / 'evaluation' / 'class_weights.json'); (GD / 'evaluation' / 'layer2_eval_utils.py').write_text(EVAL_UTILS_SRC, encoding='utf-8')
GOX_METRIC_SPEC = {**{k: v for k, v in METRIC_SPEC.items() if k not in ('report_views',)}, 'event_recall': 'NOT AVAILABLE: no event/group/entity identifier exists',
                   'report_views': ['full split (primary)', 'rows whose exact vector is NOT in train', 'rows whose matching train vector is label-mixed (excluded view)', 'per sequential block (10 blocks) to expose drift']}
dump_json(GOX_METRIC_SPEC, GD / 'evaluation' / 'metric_spec.json')
dump_json({'n_blocks': CFG['mtgox_blocks'], 'rule': 'np.array_split of each eval split in file order', 'positive_rate_over_10_blocks_all_rows': prate, 'purpose': 'expose label/feature drift; NOT a substitute for a time split'}, GD / 'evaluation' / 'eval_blocks.json')
dump_json({'source_row_index_bounds': GOX_BOUNDS, 'rule': 'sequential SOURCE ROW-INDEX split 70/15/15 (int(n*.70), int(n*.85)); source_row_id of split row i = lower bound + i', 'claimed_time_safe': bool(CHRONO_SUPPORTED), 'claimed_entity_independent': False}, GD / 'model_data' / 'split_definition.json')
GOX_PREP_REF = {'artifact': str(L2_ROOT / 'mtgox_metadata.joblib'), 'sha256': FROZEN_PIN['mtgox_metadata.joblib']['sha256'], 'fitted_on': 'TRAIN partition only (numpy recomputation rel diff %.1e vs all-partition %.1e)' % (g_train, g_all),
                'already_applied_to_frozen_parquets': True, 'refit_policy': 'NEVER refit the frozen preprocessing.', 'imputer_statistics': GOX_M['imputer'].statistics_.tolist(), 'scaler_center': GOX_M['scaler'].center_.tolist(),
                'scaler_scale': GOX_M['scaler'].scale_.tolist(), 'feature_order': GOX_FEATS}
dump_json(GOX_PREP_REF, GD / 'preprocessing' / 'preprocessing_reference.json')
dump_json({'features_config': {'A_original_verified': {'columns': list(GOX_FEATS), 'n': len(GOX_FEATS)}}, 'note': 'no identity-reduction config: no participant/coin identifier exists; features are 4 raw activity/size fields + 3 derived'}, GD / 'model_data' / 'feature_configs.json')
if GOX_IMPROVED is not None:
    GOX_IMPROVED['manifest'].to_parquet(GD / 'model_data' / 'improved_chronological_split_manifest.parquet', index=False)
    joblib.dump({'imputer': GOX_IMPROVED['imputer'], 'scaler': GOX_IMPROVED['scaler'], 'features': GOX_FEATS, 'fitted_on': 'improved-split TRAIN only'}, GD / 'preprocessing' / 'improved_split_imputer_scaler.joblib')
GOX_REPORT = {'classification': GOX_CLASS_NAME, 'verdict': GOX_VERDICT, 'reasons': GOX_REASONS, 'source_columns': hdr, 'column_roles': roles.reset_index().to_dict(orient='records'), 'documentation_files': [str(p.relative_to(BASE)) for p in DOC_FILES],
              'documentation_keyword_lines': GOX_DOC_HITS, 'time_columns_evidence': TIME_EVID, 'chronology_supported': CHRONO_SUPPORTED, 'entity_id_candidates_by_name': ENT_EVID, 'participant_like_columns': ENTITY_CANDIDATES, 'generic_id_columns': GENERIC_ID_COLS,
              'hidden_column_label_association': HID.reset_index().to_dict(orient='records'), 'model_feature_single_AUC': FEAT_AUC, 'near_deterministic_hidden_predictors': HID_DET,
              'repeated_vector_structure': GOX_REPEAT, 'within_conflict_hidden_column_auc': CONFLICT_EXPL, 'conflict_explaining_columns': EXPLAINERS, 'positive_rate_by_block': prate,
              'corrections': GOX_CHECK_EXCL, 'improved_split': ({'time_column': GOX_IMPROVED['time_column'], 'counts': GOX_IMPROVED['counts'], 'positive_rate': GOX_IMPROVED['positive_rate']} if GOX_IMPROVED else None)}
dump_json(GOX_REPORT, GD / 'reports' / 'label_provenance_and_classification_report.json')
print(f'staged files written in {time.time()-t0:.0f}s')

  wrote sapienza/metadata/sapienza_train_sidecar.parquet  (600827, 17)  15.5 MB
  wrote sapienza/metadata/sapienza_val_sidecar.parquet  (121255, 17)  3.8 MB
  wrote sapienza/metadata/sapienza_test_sidecar.parquet  (89334, 17)  2.8 MB
staged files written in 8s


In [ ]:
# ---------------- verdict inputs, manifests ----------------
COIN_OVERLAP_TXT = ', '.join(f"{r['split']} {r['rows_coin_seen_in_train']:.0%}" for r in COIN.get('overlap', []))
SAP_LIMITS = [f"very few positives: {SAP_FACTS['positive_rows']} rows across {SAP_FACTS['positive_groups']} positive event groups (train/val/test); metrics at row level are unstable, report event-level results with wide intervals",
              (f"coin overlap: share of rows whose coin occurs in train = {COIN_OVERLAP_TXT}; absolute price-level features may identify coins (see feature config B)" if PRIMARY_COIN else 'no coin identifier in the source: coin-identity leakage cannot be tested'),
              f"exact-vector overlap with train: val {LEAK['val']['share_vector_in_train']:.1%}, test {LEAK['test']['share_vector_in_train']:.1%} of rows; positives with a train-identical vector: val {LEAK['val']['positives_vector_in_train']}, test {LEAK['test']['positives_vector_in_train']}. Repeated vectors are retained; use the 'novel-vector' reporting view as a sensitivity analysis",
              'features are 5-second pump-window statistics of altcoin pump events: NOT compatible with Binance BTC/ETH 1-minute features; benchmark only, never deployable on the six-feature Binance space',
              'label semantics (`gt`, `pump_index`) rely on the dataset README (keyword lines are in the report) - human confirmation required',
              'Sapienza `rush_order` statistics are source-defined and must not be equated with Binance rush_order_density']
SAP_BLOCKERS = [f'no positive rows/groups in {sp}' for sp in SPL if SAP_FACTS['positive_rows'][sp] == 0 or SAP_FACTS['positive_groups'][sp] == 0]
SAP_WEAK = [sp for sp in SPL if SAP_FACTS['positive_groups'][sp] < CFG['min_pos_groups']]
if SAP_WEAK: SAP_LIMITS.append(f'fewer than {CFG["min_pos_groups"]} positive event groups in {SAP_WEAK}: event-level conclusions will be statistically weak')
GOX_LIMITS = [f'classification {GOX_CLASS_NAME}'] + GOX_REASONS + ['features are 4 raw fields + 3 derived (trade-count ratio, log bitcoins, |price deviation|); not compatible with Binance live features; benchmark only']

def frozen_ref(name): return {'path': str(L2_ROOT / name), 'sha256': FROZEN_PIN[name]['sha256'], 'size_bytes': FROZEN_PIN[name]['size']}
def xgb_recommended(weights):
    return {'status': 'STARTING POINT - not tuned; tune on VALIDATION only', 'objective': 'binary:logistic', 'eval_metric': ['aucpr', 'logloss'], 'tree_method': 'hist', 'learning_rate': 0.05, 'max_depth': 4, 'min_child_weight': 5,
            'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 5.0, 'max_delta_step': 1, 'n_estimators': 2000, 'early_stopping_rounds': 100, 'early_stopping_data': 'validation split, metric aucpr', 'random_state': SEED, 'n_jobs': -1,
            'scale_pos_weight_candidates': {'none': 1.0, 'sqrt': weights['scale_pos_weight_sqrt'], 'full': weights['scale_pos_weight_full']}, 'resampling': 'none (no SMOTE / under- / over-sampling)'}
LOAD_RECIPE = ['import json, pandas as pd; m = json.load(open(MANIFEST))', 'X_train = pd.read_parquet(m["splits"]["train"]["parquet"])[m["features"]["configs"][CONFIG]["columns"]]  # same for val/test',
               'y_train = pd.read_parquet(m["splits"]["train"]["parquet"])[m["target_column"]]', 'sidecar_train = pd.read_parquet(OUT_ROOT / m["splits"]["train"]["sidecar"])  # row i of the sidecar == row i of the parquet (assert on row_id)',
               'features are ALREADY imputed+scaled with the frozen train-only preprocessing: do NOT refit']

SAP_MANIFEST = {'manifest_version': '1.0', 'dataset': 'sapienza', 'output_root': str(OUT_FINAL), 'created_utc': datetime.now(timezone.utc).isoformat(), 'created_by': '01_layer2_model_ready_v1.ipynb', 'verdict': 'PENDING_VALIDATION',
    'source_provenance': {'raw_source': SOURCE_PIN[SAP_FILE.name], 'documentation_files': [str(p.relative_to(BASE)) for p in scan_docs(SAP_DIR)], 'frozen_pipeline': '02_layer2_final_pipeline_and_audit.ipynb (sapienza_split: chronological, pump_index-group-disjoint, 1h embargo)',
                          'replay_verification': SAP_FACTS['alignment'], 'train_only_preprocessing_verified': True},
    'target_column': SAP_T, 'target_semantics': 'gt = ground-truth pump label per 5-second window (definition per dataset README; human confirmation pending)',
    'features': {'order': SAP_FEATS, 'n': len(SAP_FEATS), 'configs': {k: v['columns'] for k, v in FEATURE_CONFIGS.items() if isinstance(v, dict) and 'columns' in v}, 'default_config': 'A_original_verified', 'config_rules': FEATURE_CONFIGS, 'metadata_columns_NOT_features': ['date', 'pump_index', 'event_group_id', 'coin', 'source_row_id', 'row_id', 'vec_hash', 'dup_category', 'flags']},
    'splits': {sp: {'parquet': str(L2_ROOT / f'sapienza_{sp}.parquet'), 'sha256': FROZEN_PIN[f'sapienza_{sp}.parquet']['sha256'], 'rows': int(len(SAP_PQ[sp])), 'positives': int(SAP_PQ[sp][SAP_T].sum()), 'negatives': int((1 - SAP_PQ[sp][SAP_T]).sum()),
                     'sidecar': f'sapienza/metadata/sapienza_{sp}_sidecar.parquet', 'event_groups': SAP_FACTS['event_groups'][sp], 'positive_groups': SAP_FACTS['positive_groups'][sp]} for sp in SPL},
    'frozen_artifacts': {n: frozen_ref(n) for n in FROZEN_PIN if n.startswith('sapienza')},
    'preprocessing': {'reference': 'sapienza/preprocessing/preprocessing_reference.json', 'frozen_joblib': frozen_ref('sapienza_metadata.joblib'), 'refit': 'never'},
    'class_imbalance': {'weights_file': 'sapienza/evaluation/class_weights.json', **SAP_W}, 'training_filter': {'default': 'none (unfiltered primary benchmark)', 'optional_file': 'sapienza/model_data/train_filter_identical_source_records.parquet' if FILTER_JUSTIFIED else None, 'rows_excluded': int(len(EXCL)), 'rule': DUP_DECISION['filter_rule']},
    'coin_generalization': {'chronological_event_disjoint_benchmark': 'unchanged (official)', 'coin_unseen_subsets': 'sidecar column coin_unseen_eval_subset (val/test)', 'coin_grouped_folds_train': 'sapienza/evaluation/coin_grouped_folds_train.parquet' if FOLDS is not None else None, 'details': FOLD_INFO},
    'leakage_precautions': ['splits are chronological, pump_index-group disjoint and separated by >= 1 h embargo (re-verified bit-for-bit)', 'imputer/scaler fitted on train only; never refit', 'no validation/test label used to select rows, features or weights',
                            'class weights from train labels only', 'feature selection (A vs B) on validation / coin-grouped train CV only; test untouched', 'metadata columns are never model inputs', 'duplicates retained; sensitivity views provided instead of deletion'],
    'recommended_xgboost': xgb_recommended(SAP_W['primary_unfiltered_train']), 'evaluation_protocol': METRIC_SPEC, 'known_limitations': SAP_LIMITS, 'loading_recipe': LOAD_RECIPE}
GOX_MANIFEST = {'manifest_version': '1.0', 'dataset': 'mtgox', 'output_root': str(OUT_FINAL), 'created_utc': datetime.now(timezone.utc).isoformat(), 'created_by': '01_layer2_model_ready_v1.ipynb', 'verdict': 'PENDING_VALIDATION',
    'classification': GOX_CLASS_NAME, 'source_provenance': {'raw_source': SOURCE_PIN[GOX_FILE.name], 'documentation_files': [str(p.relative_to(BASE)) for p in DOC_FILES], 'replay_verification': GOX_ALIGN, 'train_only_preprocessing_verified': True},
    'target_column': GOX_T, 'target_semantics': 'wash = wash-trading label; definition/derivation NOT documented in the available source (see label_provenance report)',
    'features': {'order': GOX_FEATS, 'n': len(GOX_FEATS), 'configs': {'A_original_verified': list(GOX_FEATS)}, 'default_config': 'A_original_verified'},
    'splits': {sp: {'parquet': str(L2_ROOT / f'mtgox_{sp}.parquet'), 'sha256': FROZEN_PIN[f'mtgox_{sp}.parquet']['sha256'], 'rows': GOX_N[sp], 'positives': int(GOX_PQ[sp][GOX_T].sum()), 'negatives': int((1 - GOX_PQ[sp][GOX_T]).sum()),
                     'row_flags': f'mtgox/evaluation/row_flags_{sp}.parquet', 'source_row_index_range': list(GOX_BOUNDS[sp])} for sp in SPL},
    'split_definition': 'mtgox/model_data/split_definition.json', 'frozen_artifacts': {n: frozen_ref(n) for n in FROZEN_PIN if n.startswith('mtgox')},
    'preprocessing': {'reference': 'mtgox/preprocessing/preprocessing_reference.json', 'frozen_joblib': frozen_ref('mtgox_metadata.joblib'), 'refit': 'never (frozen)'},
    'class_imbalance': {'weights_file': 'mtgox/evaluation/class_weights.json', **GOX_W}, 'training_filter': {'default': 'none', 'reason': GOX_CHECK_EXCL['reason']},
    'row_flags_columns': {'vector_seen_in_train': 'eval rows whose exact feature vector occurs in train', 'train_cluster_label_mixed': 'the vector\'s TRAIN rows carry both labels (computed from train labels only)', 'n_same_vector_in_train': 'count of identical train vectors'},
    'improved_split': ({'manifest': 'mtgox/model_data/improved_chronological_split_manifest.parquet', 'preprocessing': 'mtgox/preprocessing/improved_split_imputer_scaler.joblib', 'time_column': GOX_IMPROVED['time_column']} if GOX_IMPROVED else None),
    'claims': {'time_safe_split': bool(CHRONO_SUPPORTED), 'entity_independent_split': False, 'label_provenance_documented': bool(DOC_FILES)},
    'leakage_precautions': ['preprocessing fitted on train only (verified), never refit', 'no label-based row removal or relabelling; conflicting clusters retained', 'class weights from train labels only', 'row-index split NOT claimed chronological or entity-independent unless evidence says so (see claims)'],
    'recommended_xgboost': xgb_recommended(GOX_W['train']), 'evaluation_protocol': GOX_METRIC_SPEC, 'known_limitations': GOX_LIMITS, 'loading_recipe': [r.replace('sidecar_train = pd.read_parquet(OUT_ROOT / m["splits"]["train"]["sidecar"])  # row i of the sidecar == row i of the parquet (assert on row_id)', 'flags_val = pd.read_parquet(OUT_ROOT / m["splits"]["val"]["row_flags"])  # row i == row i of the parquet') for r in LOAD_RECIPE]}
dump_json(SAP_MANIFEST, SD / 'training_manifest.json'); dump_json(GOX_MANIFEST, GD / 'training_manifest.json')
(STAGE / 'README.md').write_text(f"""# processed_layer2_model_ready_v1
Created {RUN_STARTED} by 01_layer2_model_ready_v1.ipynb. FROZEN inputs (never modified): `processed_layer2_final_v3/` + raw Layer 2 sources.
Frozen parquets are REFERENCED (path + SHA-256) by each `training_manifest.json`, not copied.

* `sapienza/training_manifest.json` - everything the XGBoost notebook needs for Sapienza (verdict, features/configs, splits, sidecars, weights, protocol, limitations)
* `mtgox/training_manifest.json` - same for Mt.Gox (classification: {GOX_CLASS_NAME})
* `*/metadata/` (Sapienza sidecars, row-aligned by `row_id`), `*/evaluation/` (weights, metric spec, `layer2_eval_utils.py`, folds/flags), `*/reports/` (evidence), `*/preprocessing/` (reference to frozen train-only preprocessing; no refit)

The two datasets must stay separate; neither is compatible with the Binance six-feature space.
""", encoding='utf-8')
print('manifests + README staged')

manifests + README staged


## H · Targeted validation of the NEW artifacts only (assertions; publish only if all pass)

In [ ]:
VAL = []
def vcheck(name, cond, detail=''):
    ok = bool(cond); VAL.append({'check': name, 'ok': ok, 'detail': detail}); print(('✅' if ok else '❌'), name, ('- ' + detail) if detail and not ok else '')
    return ok
R = STAGE                                                                                  # artifacts are read back from disk (staging path)

# ---- frozen data untouched ----
now = {p.name: {'size': p.stat().st_size, 'mtime_ns': p.stat().st_mtime_ns, 'sha256': sha256_file(p)} for p in L2_ROOT.iterdir() if p.is_file()}
vcheck('frozen v3: identical file set', sorted(now) == FROZEN_LISTING, f'{sorted(set(now) ^ set(FROZEN_LISTING))}')
vcheck('frozen v3: SHA-256 / size / mtime unchanged', all(now[n] == FROZEN_PIN[n] for n in FROZEN_PIN if n in now))
vcheck('raw sources unchanged (size, mtime)', all((Path(v['path']).stat().st_size, Path(v['path']).stat().st_mtime_ns) == (v['size'], v['mtime_ns']) for v in SOURCE_PIN.values()))
vcheck('no frozen file copied into output (no *.joblib in sapienza/, only improved-split joblib in mtgox/ if created)',
       not list((R / 'sapienza').rglob('*.joblib')) and {p.name for p in (R / 'mtgox').rglob('*.joblib')} <= {'improved_split_imputer_scaler.joblib'})

# ---- Sapienza sidecars ----
srcv = SRC_DATE_CHECK.copy(); srcv['date'] = pd.to_datetime(srcv['date'], utc=True); srcv['pump_index'] = pd.to_numeric(srcv['pump_index'], errors='coerce')
side = {sp: pd.read_parquet(R / 'sapienza' / 'metadata' / f'sapienza_{sp}_sidecar.parquet') for sp in SPL}
for sp in SPL:
    s, fz = side[sp], pd.read_parquet(L2_ROOT / f'sapienza_{sp}.parquet')
    vcheck(f'sidecar {sp}: row count == frozen parquet', len(s) == len(fz))
    vcheck(f'sidecar {sp}: row_id == positional index', np.array_equal(s.row_id.to_numpy(), np.arange(len(fz))))
    vcheck(f'sidecar {sp}: label == frozen parquet target', np.array_equal(s['gt'].to_numpy().astype(np.int8), fz[SAP_T].to_numpy().astype(np.int8)))
    vcheck(f'sidecar {sp}: chronological order preserved', pd.Series(pd.to_datetime(s.date, utc=True)).is_monotonic_increasing)
    o = srcv.iloc[s.source_row_id.to_numpy()]
    vcheck(f'sidecar {sp}: date/pump_index/label equal the RAW SOURCE at source_row_id', (pd.to_datetime(s.date, utc=True).to_numpy() == o.date.to_numpy()).all() and np.array_equal(np.nan_to_num(s.pump_index.to_numpy(float), nan=-9), np.nan_to_num(o.pump_index.to_numpy(float), nan=-9)) and np.array_equal(s['gt'].to_numpy(), o['gt'].to_numpy()))
    vcheck(f'sidecar {sp}: vec_hash == hash of frozen features', np.array_equal(s.vec_hash.to_numpy(np.uint64), hash_rows(fz, SAP_FEATS)))
    vcheck(f'sidecar {sp}: feature columns absent from sidecar', not (set(SAP_FEATS) & set(s.columns)))
vcheck('sidecars: source_row_id unique across splits', pd.concat([side[sp].source_row_id for sp in SPL]).is_unique)
vcheck('sidecars: pump_index groups disjoint across splits', not (set(side['train'].event_group_id.dropna()) & set(side['val'].event_group_id.dropna())) and not (set(side['train'].event_group_id.dropna()) & set(side['test'].event_group_id.dropna())) and not (set(side['val'].event_group_id.dropna()) & set(side['test'].event_group_id.dropna())))
vcheck('sidecars: train < val < test dates with >= 1h embargo', pd.to_datetime(side['train'].date, utc=True).max() + EMBARGO <= pd.to_datetime(side['val'].date, utc=True).min() and pd.to_datetime(side['val'].date, utc=True).max() + EMBARGO <= pd.to_datetime(side['test'].date, utc=True).min())
tr_h = set(side['train'].vec_hash)
vcheck('flag vec_seen_in_train recomputed independently', all(np.array_equal(side[sp].vec_seen_in_train.to_numpy(), side[sp].vec_hash.isin(tr_h).to_numpy()) for sp in ('val', 'test')))
vcheck('positives with train-identical vector == reported', all(int((side[sp].vec_seen_in_train & (side[sp]['gt'] == 1)).sum()) == LEAK[sp]['positives_vector_in_train'] for sp in ('val', 'test')))
if PRIMARY_COIN:
    trc2 = set(side['train'].coin.dropna())
    vcheck('flag coin_seen_in_train recomputed independently', all(np.array_equal(side[sp].coin_seen_in_train.to_numpy(), side[sp].coin.isin(trc2).fillna(False).to_numpy()) for sp in ('val', 'test')))
    vcheck('coin_unseen_eval_subset == ~coin_seen_in_train (val/test)', all(np.array_equal(side[sp].coin_unseen_eval_subset.to_numpy(), ~side[sp].coin_seen_in_train.to_numpy()) for sp in ('val', 'test')))

# ---- feature configs / order ----
fc = json.loads((R / 'sapienza' / 'model_data' / 'feature_configs.json').read_text()); fz_cols = list(pd.read_parquet(L2_ROOT / 'sapienza_train.parquet').columns)
vcheck('config A == frozen feature order', fc['A_original_verified']['columns'] == fz_cols[:-1] == list(SAP_M['features']))
vcheck('config B is an order-preserving subset of A', [c for c in fc['A_original_verified']['columns'] if c in fc['B_price_identity_reduced']['columns']] == fc['B_price_identity_reduced']['columns'])
vcheck('config B drops only price-level features', set(fc['B_price_identity_reduced'].get('dropped', [])) <= set(PRICE_LEVEL))

# ---- no preprocessing refit ----
pr = json.loads((R / 'sapienza' / 'preprocessing' / 'preprocessing_reference.json').read_text()); jb = joblib.load(L2_ROOT / 'sapienza_metadata.joblib')
vcheck('Sapienza preprocessing reference == frozen joblib (sha + parameters)', pr['sha256'] == sha256_file(L2_ROOT / 'sapienza_metadata.joblib') and np.allclose(pr['imputer_statistics'], jb['imputer'].statistics_) and np.allclose(pr['scaler_scale'], jb['scaler'].scale_) and np.allclose(pr['scaler_center'], jb['scaler'].center_))
gjb = joblib.load(L2_ROOT / 'mtgox_metadata.joblib'); gpr = json.loads((R / 'mtgox' / 'preprocessing' / 'preprocessing_reference.json').read_text())
vcheck('Mt.Gox preprocessing reference == frozen joblib', gpr['sha256'] == sha256_file(L2_ROOT / 'mtgox_metadata.joblib') and np.allclose(gpr['scaler_scale'], gjb['scaler'].scale_) and np.allclose(gpr['imputer_statistics'], gjb['imputer'].statistics_))

# ---- class weights recomputed from frozen labels ----
for ds, dsname, tgt in (('sapienza', 'sapienza', SAP_T), ('mtgox', 'mtgox', GOX_T)):
    y = pd.read_parquet(L2_ROOT / f'{dsname}_train.parquet', columns=[tgt])[tgt].to_numpy().astype(int); cw = json.loads((R / ds / 'evaluation' / 'class_weights.json').read_text())
    key = 'primary_unfiltered_train' if ds == 'sapienza' else 'train'; neg, pos = int((1 - y).sum()), int(y.sum())
    vcheck(f'{ds}: class weights recomputed from TRAIN labels only', cw[key]['positives'] == pos and cw[key]['negatives'] == neg and abs(cw[key]['scale_pos_weight_full'] - neg / pos) < 1e-9 and abs(cw[key]['scale_pos_weight_full'] - cw['frozen_metadata_scale_pos_weight']) < 1e-6 * neg / pos and abs(cw[key]['scale_pos_weight_sqrt'] - math.sqrt(neg / pos)) < 1e-9)
if FILTER_JUSTIFIED:
    ex = pd.read_parquet(R / 'sapienza' / 'model_data' / 'train_filter_identical_source_records.parquet'); sref = pd.Series(SREC_HASH)
    vcheck('train filter: rows are TRAIN rows only and identical-source-record duplicates', ex.row_id.max() < len(side['train']) and (sref[ex.source_row_id.to_numpy()].map(sref.value_counts()) >= 2).all() and set(ex.source_row_id) <= set(side['train'].source_row_id))
    vcheck('train filter: reasons valid and counts consistent with report', set(ex.reason) <= {'R1_train_internal_identical_source_record', 'R2_train_copy_of_record_present_in_eval'} and len(ex) == DUP_DECISION['rows_excluded'])
    cwf = json.loads((R / 'sapienza' / 'evaluation' / 'class_weights.json').read_text())['filtered_train_option']; yk = side['train']['gt'].to_numpy()[~side['train'].row_id.isin(ex.row_id).to_numpy()]
    vcheck('train filter: filtered class weights recomputed', cwf['positives'] == int(yk.sum()) and cwf['negatives'] == int((1 - yk).sum()))
# ---- coin-grouped folds ----
if FOLDS is not None:
    fo = pd.read_parquet(R / 'sapienza' / 'evaluation' / 'coin_grouped_folds_train.parquet'); m = fo.merge(side['train'][['row_id', 'coin', 'event_group_id', 'gt']], on='row_id', suffixes=('', '_s'))
    vcheck('folds: cover every train row exactly once', len(fo) == len(side['train']) and fo.row_id.is_unique and set(fo.row_id) == set(side['train'].row_id))
    vcheck('folds: each coin in exactly one fold', m.fillna({'coin': '__NA__'}).groupby('coin')['fold'].nunique().max() == 1)
    vcheck('folds: each event group in exactly one fold', m.dropna(subset=['event_group_id']).groupby('event_group_id')['fold'].nunique().max() == 1)
    cwj = json.loads((R / 'sapienza' / 'evaluation' / 'class_weights.json').read_text())['coin_fold_train_weights']
    vcheck('folds: per-fold TRAIN weights recomputed from fold-complement labels', all(cwj[str(f)]['positives'] == int(m.loc[m.fold != f, 'gt'].sum()) and cwj[str(f)]['negatives'] == int((1 - m.loc[m.fold != f, 'gt']).sum()) for f in sorted(m.fold.unique())))
    vcheck('folds: every fold has >=1 positive', all(m.loc[m.fold == f, 'gt'].sum() > 0 for f in m.fold.unique()))

# ---- Mt.Gox flags ----
for i, sp in enumerate(SPL):
    fl = pd.read_parquet(R / 'mtgox' / 'evaluation' / f'row_flags_{sp}.parquet'); vcheck(f'mtgox flags {sp}: row count == frozen parquet', len(fl) == GOX_N[sp])
trh2 = pd.DataFrame({'h': hash_rows(pd.read_parquet(L2_ROOT / 'mtgox_train.parquet'), GOX_FEATS), 'y': pd.read_parquet(L2_ROOT / 'mtgox_train.parquet', columns=[GOX_T])[GOX_T].to_numpy()})
trg = trh2.groupby('h')['y'].agg(['size', 'sum']); veval = pd.read_parquet(L2_ROOT / 'mtgox_val.parquet'); hv = pd.Series(hash_rows(veval, GOX_FEATS)); flv = pd.read_parquet(R / 'mtgox' / 'evaluation' / 'row_flags_val.parquet')
vcheck('mtgox flags val: vector_seen_in_train recomputed independently', np.array_equal(flv.vector_seen_in_train.to_numpy(), hv.isin(trg.index).to_numpy()))
mixed_set = set(trg.index[(trg['sum'] > 0) & (trg['sum'] < trg['size'])]); vcheck('mtgox flags val: train_cluster_label_mixed recomputed independently', np.array_equal(flv.train_cluster_label_mixed.to_numpy(), hv.isin(mixed_set).to_numpy()))
del trh2, hv, veval

✅ frozen v3: identical file set 
✅ frozen v3: SHA-256 / size / mtime unchanged 
✅ raw sources unchanged (size, mtime) 
✅ no frozen file copied into output (no *.joblib in sapienza/, only improved-split joblib in mtgox/ if created) 
✅ sidecar train: row count == frozen parquet 
✅ sidecar train: row_id == positional index 
✅ sidecar train: label == frozen parquet target 
✅ sidecar train: chronological order preserved 
✅ sidecar train: date/pump_index/label equal the RAW SOURCE at source_row_id 
✅ sidecar train: vec_hash == hash of frozen features 
✅ sidecar train: feature columns absent from sidecar 
✅ sidecar val: row count == frozen parquet 
✅ sidecar val: row_id == positional index 
✅ sidecar val: label == frozen parquet target 
✅ sidecar val: chronological order preserved 
✅ sidecar val: date/pump_index/label equal the RAW SOURCE at source_row_id 
✅ sidecar val: vec_hash == hash of frozen features 
✅ sidecar val: feature columns absent from sidecar 
✅ sidecar test: row count == froze

In [ ]:
# ---- metrics module: unit tests on the SAVED file (imported from disk) ----
import importlib.util
sys.dont_write_bytecode = True
spec = importlib.util.spec_from_file_location('layer2_eval_utils_saved', R / 'sapienza' / 'evaluation' / 'layer2_eval_utils.py'); mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
vcheck('eval utils: saved file identical to the module exercised in this notebook', (R / 'sapienza' / 'evaluation' / 'layer2_eval_utils.py').read_text() == (R / 'mtgox' / 'evaluation' / 'layer2_eval_utils.py').read_text() == EVAL_UTILS_SRC)
y = np.array([1, 1, 0, 0, 0, 0, 0, 0, 0, 0]); s = np.array([.9, .4, .8, .3, .2, .1, .1, .1, .05, .0]); m = mod.rare_event_metrics(y, s, 0.5)
vcheck('eval utils: confusion matrix / precision / recall / F1 / FP per 1000', (m['tp'], m['fp'], m['fn'], m['tn']) == (1, 1, 1, 7) and abs(m['precision'] - .5) < 1e-12 and abs(m['recall'] - .5) < 1e-12 and abs(m['f1'] - .5) < 1e-12 and abs(m['false_positives_per_1000_observations'] - 100.0) < 1e-9 and m['confusion_matrix'] == [[7, 1], [1, 1]])
vcheck('eval utils: PR-AUC / ROC-AUC match sklearn', abs(m['pr_auc'] - __import__('sklearn.metrics', fromlist=['x']).average_precision_score(y, s)) < 1e-12 and abs(m['roc_auc'] - __import__('sklearn.metrics', fromlist=['x']).roc_auc_score(y, s)) < 1e-12)
g = np.array([0, 0, -1, -1, -1, -1, -1, -1, -1, -1]); me = mod.rare_event_metrics(y, s, 0.5, groups=g); vcheck('eval utils: event-level recall (1 of 1 group detected)', me['event_groups_positive'] == 1 and me['event_groups_detected'] == 1 and me['event_recall'] == 1.0)
g2 = np.array([0, 1, -1, -1, -1, -1, -1, -1, -1, -1]); me2 = mod.rare_event_metrics(y, s, 0.5, groups=g2); vcheck('eval utils: event-level recall (1 of 2 groups detected)', me2['event_groups_positive'] == 2 and me2['event_recall'] == 0.5)
rng_t = np.random.default_rng(0); yy = (rng_t.random(5000) < .02).astype(int); ss = rng_t.random(5000) + yy * .8; th = mod.choose_threshold_on_validation(yy, ss, max_fp_per_1000=20); mm = mod.rare_event_metrics(yy, ss, th)
vcheck('eval utils: validation threshold respects FP budget', mm['false_positives_per_1000_observations'] <= 20 + 1e-9 and mm['recall'] > 0.3)
vcheck('eval utils: single-class input does not crash (NaN AUCs)', np.isnan(mod.rare_event_metrics(np.zeros(5), np.arange(5.), 2.0)['pr_auc']))

# ---- manifests: readable, paths exist, hashes match, honest claims ----
for ds, mf in (('sapienza', 'sapienza/training_manifest.json'), ('mtgox', 'mtgox/training_manifest.json')):
    M = json.loads((R / mf).read_text()); feats = M['features']['order']
    vcheck(f'{ds} manifest: feature order == frozen parquet columns', feats == list(pd.read_parquet(Path(M['splits']['train']['parquet'])).columns[:-1]))
    vcheck(f'{ds} manifest: split parquets exist, SHA-256 and row counts match', all(Path(v['parquet']).is_file() and sha256_file(Path(v['parquet'])) == v['sha256'] and pq.ParquetFile(v['parquet']).metadata.num_rows == v['rows'] for v in M['splits'].values()))
    vcheck(f'{ds} manifest: every referenced artifact exists', all((R / v).exists() for sp in M['splits'].values() for k, v in sp.items() if k in ('sidecar', 'row_flags')) and (R / M['preprocessing']['reference']).exists() and (R / M['class_imbalance']['weights_file']).exists())
    vcheck(f'{ds} manifest: class counts consistent with parquet', all(v['positives'] + v['negatives'] == v['rows'] for v in M['splits'].values()))
    vcheck(f'{ds} manifest: required sections present', all(k in M for k in ('dataset', 'source_provenance', 'target_column', 'features', 'splits', 'preprocessing', 'class_imbalance', 'leakage_precautions', 'recommended_xgboost', 'evaluation_protocol', 'known_limitations', 'loading_recipe')))
Mg = json.loads((R / 'mtgox/training_manifest.json').read_text())
vcheck('mtgox manifest: time-safe claim only if a reliable time column exists; never claims entity independence', Mg['claims']['time_safe_split'] == bool(CHRONO_SUPPORTED) and Mg['claims']['entity_independent_split'] is False)
vcheck('datasets kept separate (no combined matrix, disjoint feature spaces)', not any('combined' in str(p).lower() for p in R.rglob('*')) and set(SAP_FEATS).isdisjoint(set(GOX_FEATS)))
vcheck('all staged parquet/json artifacts readable', all((pq.ParquetFile(p).metadata.num_rows >= 0) for p in R.rglob('*.parquet')) and all(json.loads(p.read_text()) is not None for p in R.rglob('*.json')))
VAL_DF = pd.DataFrame(VAL); n_fail = int((~VAL_DF.ok).sum()); print(f'\nTARGETED VALIDATION: {len(VAL_DF)-n_fail}/{len(VAL_DF)} passed')
if n_fail: display(VAL_DF[~VAL_DF.ok]); raise AssertionError(f'{n_fail} validation check(s) FAILED - NOT published. Inspect staging dir: {STAGE}')

✅ eval utils: saved file identical to the module exercised in this notebook 
✅ eval utils: confusion matrix / precision / recall / F1 / FP per 1000 
✅ eval utils: PR-AUC / ROC-AUC match sklearn 
✅ eval utils: event-level recall (1 of 1 group detected) 
✅ eval utils: event-level recall (1 of 2 groups detected) 
✅ eval utils: validation threshold respects FP budget 
✅ eval utils: single-class input does not crash (NaN AUCs) 
✅ sapienza manifest: feature order == frozen parquet columns 
✅ sapienza manifest: split parquets exist, SHA-256 and row counts match 
✅ sapienza manifest: every referenced artifact exists 
✅ sapienza manifest: class counts consistent with parquet 
✅ sapienza manifest: required sections present 
✅ mtgox manifest: feature order == frozen parquet columns 
✅ mtgox manifest: split parquets exist, SHA-256 and row counts match 
✅ mtgox manifest: every referenced artifact exists 
✅ mtgox manifest: class counts consistent with parquet 
✅ mtgox manifest: required sections pre

In [ ]:
# ---------------- publish (atomic rename) + final verdicts ----------------
SAP_VERDICT = 'BLOCKED' if SAP_BLOCKERS else 'READY'
for ds, verdict in (('sapienza', SAP_VERDICT), ('mtgox', GOX_VERDICT)):
    p = STAGE / ds / 'training_manifest.json'; M = json.loads(p.read_text()); M['verdict'] = verdict; M['validation'] = {'checks': len(VAL_DF), 'passed': int(VAL_DF.ok.sum()), 'run_utc': datetime.now(timezone.utc).isoformat()}; dump_json(M, p)
dump_json({'run_started_utc': RUN_STARTED, 'run_finished_utc': datetime.now(timezone.utc).isoformat(), 'checks': VAL, 'verdicts': {'sapienza': SAP_VERDICT, 'mtgox': GOX_VERDICT}}, STAGE / 'VALIDATION_REPORT.json')
assert not OUT_FINAL.exists(); STAGE.rename(OUT_FINAL)
assert OUT_FINAL.is_dir() and not STAGE.exists()
now2 = {p.name: sha256_file(p) for p in L2_ROOT.iterdir() if p.is_file()}; assert now2 == {n: v['sha256'] for n, v in FROZEN_PIN.items()}, 'frozen v3 changed after publish!'
print('PUBLISHED ->', OUT_FINAL)

PUBLISHED -> /content/drive/MyDrive/research datasets/processed_layer2_model_ready_v1


In [ ]:
# ================================= SHARE BLOCK =================================
files = sorted(p for p in OUT_FINAL.rglob('*') if p.is_file())
print('<<<<< SHARE BLOCK: Layer 2 model-ready v1 >>>>>')
print(f'1-2. OUTPUT: {OUT_FINAL}  ({len(files)} files, {sum(p.stat().st_size for p in files)/1e6:.1f} MB)')
for p in files: print(f'     {p.relative_to(OUT_FINAL)}  ({p.stat().st_size/1e3:.0f} KB)')
print('\n3. SAPIENZA corrections: alignment proven bit-for-bit', {sp: a['features_bit_identical'] and a['labels_identical'] for sp, a in ALIGN.items()}, '| sidecars written for train/val/test | coin column:', PRIMARY_COIN, '| id columns:', ID_COLS)
print('   rows (train/val/test):', {sp: len(SAP_PQ[sp]) for sp in SPL}, '| positives:', SAP_FACTS['positive_rows'], '| positive groups:', SAP_FACTS['positive_groups'])
print('   duplicates: val/test rows with train-identical vector', {sp: f"{LEAK[sp]['share_vector_in_train']:.1%}" for sp in ('val', 'test')}, '| positives with train-identical vector:', {sp: LEAK[sp]['positives_vector_in_train'] for sp in ('val', 'test')})
print('   category counts (all splits):', {k: int(v) for k, v in tab['total'].items()})
print('4. MT.GOX:', GOX_CLASS_NAME); [print('     -', r) for r in GOX_REASONS]
print('5. ROWS: Sapienza train before/after optional filter:', before['rows'], '->', after['rows'], '(excluded %d; positives excluded %d); val/test never filtered. Mt.Gox excluded: 0.' % (len(EXCL), int(EXCL['gt'].sum()) if len(EXCL) else 0))
print('   train class balance before/after:', before, after)
print('6. FEATURE CONFIGS: A =', len(cfg_A), 'columns | B =', len(cfg_B), 'columns, dropped', dropB, '| rule:', evidence_B)
print('7. IMBALANCE: weights from TRAIN only; Sapienza spw full/sqrt:', round(SAP_W['primary_unfiltered_train']['scale_pos_weight_full'], 2), round(SAP_W['primary_unfiltered_train']['scale_pos_weight_sqrt'], 2), '| Mt.Gox full:', round(GOX_W['train']['scale_pos_weight_full'], 3), '| no resampling/SMOTE')
print('8. MANIFESTS: sapienza/training_manifest.json, mtgox/training_manifest.json')
print(f'9. TARGETED VALIDATION: {int(VAL_DF.ok.sum())}/{len(VAL_DF)} passed; frozen v3 SHA-256 unchanged before and after publish')
print('10. LIMITATIONS (Sapienza):'); [print('     -', l) for l in SAP_LIMITS]
print('    LIMITATIONS (Mt.Gox):'); [print('     -', l) for l in GOX_LIMITS]
print('\nVERDICT  SAPIENZA XGBOOST TRAINING :', SAP_VERDICT, '(technical gates: bit-exact alignment, disjoint chronological groups, train-only preprocessing, validated artifacts)')
print('VERDICT  MT.GOX  XGBOOST TRAINING :', GOX_VERDICT, '->', GOX_CLASS_NAME)
print('<<<<< END SHARE BLOCK >>>>>')

<<<<< SHARE BLOCK: Layer 2 model-ready v1 >>>>>
1-2. OUTPUT: /content/drive/MyDrive/research datasets/processed_layer2_model_ready_v1  (27 files, 26.7 MB)
     README.md  (1 KB)
     VALIDATION_REPORT.json  (9 KB)
     mtgox/evaluation/class_weights.json  (0 KB)
     mtgox/evaluation/eval_blocks.json  (0 KB)
     mtgox/evaluation/layer2_eval_utils.py  (3 KB)
     mtgox/evaluation/metric_spec.json  (1 KB)
     mtgox/evaluation/row_flags_test.parquet  (141 KB)
     mtgox/evaluation/row_flags_train.parquet  (1139 KB)
     mtgox/evaluation/row_flags_val.parquet  (142 KB)
     mtgox/model_data/feature_configs.json  (0 KB)
     mtgox/model_data/split_definition.json  (0 KB)
     mtgox/preprocessing/preprocessing_reference.json  (1 KB)
     mtgox/reports/label_provenance_and_classification_report.json  (12 KB)
     mtgox/training_manifest.json  (9 KB)
     sapienza/evaluation/class_weights.json  (1 KB)
     sapienza/evaluation/coin_grouped_folds_train.parquet  (3098 KB)
     sapienza/evaluati